# 1. Setup: Packages and Global Parameters


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv
import glob
import json
import os

# Folder in Google Drive containing the batch JSON files to combine.
BATCH_DIR = "/content/drive/MyDrive/phd/P4/Data"
BATCH_PATTERN = "batch*.json"
OUTPUT_CSV = os.path.join(BATCH_DIR, "aiid_full_incidents_processed.csv")

FIELDNAMES = [
    "incident_id", "title", "description", "date", "date_modified", "year",
    "developer", "deployer", "harmed", "implicated_systems", "report_count",
    "mit_risk_domain", "mit_risk_subdomain", "mit_entity", "mit_intent", "mit_timing",
    "cset_sector", "cset_lives_lost", "cset_injuries", "cset_location_country",
    "cset_ai_system_description", "cset_entities_raw",
    "gmf_known_ai_goal", "gmf_known_technology", "gmf_known_technical_failure",
    "n_cset_annotators",
]


def names_join(items):
    return "; ".join(x.get("name", "") for x in (items or []) if x)


def find_classification(classifications, namespace, require_publish=False):
    for c in classifications or []:
        if c.get("namespace") == namespace and (not require_publish or c.get("publish")):
            return c
    return None


def count_namespace_prefix(classifications, prefix):
    return sum(1 for c in classifications or [] if str(c.get("namespace", "")).startswith(prefix))


def classification_attr(classification, short_name):
    """Look up an attribute by short_name and decode its JSON-encoded value_json."""
    if not classification:
        return None
    for a in classification.get("attributes", []):
        if a.get("short_name") == short_name:
            try:
                return json.loads(a["value_json"])
            except (TypeError, ValueError):
                return a["value_json"]
    return None


def build_row(incident):
    classifications = incident.get("classifications") or []
    mit = find_classification(classifications, "MIT")
    csetv1 = find_classification(classifications, "CSETv1")
    # GMF ("known AI goal/technology/failure") is only trustworthy once published.
    gmf = find_classification(classifications, "GMF", require_publish=True)

    date = incident.get("date") or ""
    sector = classification_attr(csetv1, "Sector of Deployment")
    lives_lost = classification_attr(csetv1, "Lives Lost")
    injuries = classification_attr(csetv1, "Injuries")

    return {
        "incident_id": incident.get("incident_id"),
        "title": incident.get("title"),
        "description": incident.get("description"),
        "date": date,
        "date_modified": incident.get("date_modified") or "",
        "year": date[:4] if date else "",
        "developer": names_join(incident.get("AllegedDeveloperOfAISystem")),
        "deployer": names_join(incident.get("AllegedDeployerOfAISystem")),
        "harmed": names_join(incident.get("AllegedHarmedOrNearlyHarmedParties")),
        "implicated_systems": names_join(incident.get("implicated_systems")),
        "report_count": len(incident.get("reports") or []),
        "mit_risk_domain": classification_attr(mit, "Risk Domain") or "",
        "mit_risk_subdomain": classification_attr(mit, "Risk Subdomain") or "",
        "mit_entity": classification_attr(mit, "Entity") or "",
        "mit_intent": classification_attr(mit, "Intent") or "",
        "mit_timing": classification_attr(mit, "Timing") or "",
        "cset_sector": "" if sector is None else str(sector),
        "cset_lives_lost": float(lives_lost) if isinstance(lives_lost, (int, float)) else "",
        "cset_injuries": float(injuries) if isinstance(injuries, (int, float)) else "",
        "cset_location_country": classification_attr(csetv1, "Location Country (two letters)") or "",
        "cset_ai_system_description": classification_attr(csetv1, "AI System Description") or "",
        # csetv1 present but attribute missing/null -> "null"; csetv1 absent entirely -> "".
        "cset_entities_raw": json.dumps(classification_attr(csetv1, "Entities")) if csetv1 is not None else "",
        "gmf_known_ai_goal": json.dumps(classification_attr(gmf, "Known AI Goal")) if gmf is not None else "",
        "gmf_known_technology": json.dumps(classification_attr(gmf, "Known AI Technology")) if gmf is not None else "",
        "gmf_known_technical_failure": json.dumps(classification_attr(gmf, "Known AI Technical Failure")) if gmf is not None else "",
        # Count of CSETv1_Annotator-* classification blocks, published or not.
        "n_cset_annotators": count_namespace_prefix(classifications, "CSETv1_Annotator-"),
    }


batch_paths = sorted(glob.glob(os.path.join(BATCH_DIR, BATCH_PATTERN)))
print(f"Found {len(batch_paths)} batch file(s)")

incidents_by_id = {}

for path in batch_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each batch file is a raw GraphQL response: {"data": {"incidents": [...]}}.
    records = next(iter(data["data"].values())) if isinstance(data, dict) and "data" in data else data

    for incident in records:
        incidents_by_id[incident["incident_id"]] = incident

    print(f"  {os.path.basename(path)}: {len(records)} record(s)")

print(f"Total unique incidents: {len(incidents_by_id)}")

rows = [build_row(inc) for inc in incidents_by_id.values()]
rows.sort(key=lambda r: r["incident_id"])

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")

# 2. Descriptive Analysis


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/phd/P4/Data/aiid_full_incidents_processed.csv"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Columns and dtypes:")
print(df.dtypes)

In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
# Summary statistics for numeric columns
df.describe(include="number").T

In [ ]:
# Value counts for object/categorical columns with manageable cardinality
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    n_unique = df[col].nunique(dropna=True)
    if 1 < n_unique <= 30:
        print(f"--- {col} ({n_unique} unique values) ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

In [ ]:
# Incidents over time, if a date-like column is present (e.g. "date")
date_cols = [c for c in df.columns if "date" in c.lower()]

if date_cols:
    date_col = date_cols[0]
    dates = pd.to_datetime(df[date_col], errors="coerce")
    print(f"Using date column: {date_col}")
    print(f"Range: {dates.min()} to {dates.max()}")
    print()
    print("Incidents per year:")
    print(dates.dt.year.value_counts().sort_index())
else:
    print("No date-like column found.")

# 3. Module 1 --- Compositional Dynamics (H1, H1a)

Tests whether the composition of realized AI incidents shifted materially
over the primary analysis window (2018-2025), and whether that shift
reflects displacement of established risk-domain categories or the
emergence of new ones alongside a stable base (H1a).


In [ ]:
import re

PRIMARY_WINDOW = (2018, 2025)
BREAK_YEAR = 2022  # pre = up to and including 2021; post = 2022 onward

df["year"] = pd.to_numeric(df["year"], errors="coerce")

primary = df[df["year"].between(*PRIMARY_WINDOW)].copy()
classified = primary[primary["mit_risk_domain"].notna() & (primary["mit_risk_domain"] != "")].copy()

# MIT risk domains are prefixed "N. <name>" (e.g. "4. Malicious Actors and Misuse");
# sort/display by that leading number rather than alphabetically.
def domain_sort_key(domain):
    m = re.match(r"\s*(\d+)", str(domain))
    return int(m.group(1)) if m else 999

domains = sorted(classified["mit_risk_domain"].unique(), key=domain_sort_key)

n_primary = len(primary)
n_classified = len(classified)
print(f"Primary window {PRIMARY_WINDOW[0]}-{PRIMARY_WINDOW[1]}: n = {n_primary}")
print(f"Classified under MIT taxonomy: n = {n_classified} ({n_classified / n_primary:.1%})")
print()
print("Risk domains:")
for d in domains:
    print(" ", d)

In [ ]:
# Table 2: annual risk-domain composition (% of classified incidents per year)
year_domain_counts = (
    classified.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)[domains]
)
year_totals = year_domain_counts.sum(axis=1)
year_domain_pct = year_domain_counts.div(year_totals, axis=0) * 100

table2 = year_domain_pct.round(1).copy()
table2["n"] = year_totals
table2

In [ ]:
# Table 3: absolute incident counts, pre- vs post-BREAK_YEAR
classified["period"] = classified["year"].apply(
    lambda y: f"Pre-{BREAK_YEAR}" if y < BREAK_YEAR else f"Post-{BREAK_YEAR}"
)

period_counts = classified.groupby(["mit_risk_domain", "period"]).size().unstack(fill_value=0)
pre_col, post_col = f"Pre-{BREAK_YEAR}", f"Post-{BREAK_YEAR}"
period_counts = period_counts.reindex(columns=[pre_col, post_col], fill_value=0).reindex(domains)

pre_total, post_total = period_counts[pre_col].sum(), period_counts[post_col].sum()
pre_pct = period_counts[pre_col] / pre_total * 100
post_pct = period_counts[post_col] / post_total * 100

table3 = pd.DataFrame({
    f"{pre_col} (n={pre_total})": period_counts[pre_col].astype(str) + " (" + pre_pct.round(1).astype(str) + "%)",
    f"{post_col} (n={post_total})": period_counts[post_col].astype(str) + " (" + post_pct.round(1).astype(str) + "%)",
    "Change (count)": period_counts[post_col] - period_counts[pre_col],
    "Change (share, pp)": (post_pct - pre_pct).round(1),
}).sort_values("Change (count)", ascending=False)

table3

In [ ]:
%pip install -q ruptures

import numpy as np
import ruptures as rpt

# Structural break test on the Malicious Actors & Misuse share series.
mam_domain = next(d for d in domains if "malicious actors" in d.lower())
mam_share = year_domain_pct[mam_domain].sort_index()
years = mam_share.index.to_numpy()
signal = mam_share.to_numpy()

# PELT selects its own number of breakpoints via a penalty (higher = fewer
# breaks); binary segmentation is asked directly for a single breakpoint.
detectors = [
    ("PELT", rpt.Pelt(model="l2").fit(signal), {"pen": 3}),
    ("Binary segmentation", rpt.Binseg(model="l2").fit(signal), {"n_bkps": 1}),
]

for algo_name, algo, kwargs in detectors:
    # predict() returns breakpoint indices, the last of which is len(signal) (end marker).
    breakpoints = [b for b in algo.predict(**kwargs) if b < len(signal)]
    break_years = [int(years[b]) for b in breakpoints]
    print(f"{algo_name}: break(s) at start of {break_years}" if break_years else f"{algo_name}: no break detected")

In [ ]:
# Zero-filled long-format helper for the Module 1 Poisson models.
#
# classified.groupby(["year", "mit_risk_domain"]).size().reset_index(name="count")
# only creates rows for year-domain cells with >=1 incident; zero-count
# cells are silently dropped from the resulting long table instead of
# entering the Poisson model as count=0. In the primary window this drops
# three real zero-count cells (2018 Misinformation, 2019 Socioeconomic &
# Environmental Harms, 2020 Human-Computer Interaction), biasing the
# affected domains' growth factors and the LR statistics. This helper
# builds the same long-format table via a zero-filled wide pivot first, so
# every year-domain cell is present.
def zero_filled_long(frame, start_year, end_year):
    """Year x domain counts in long format, INCLUDING zero-count cells."""
    wide = (
        frame.groupby(["year", "mit_risk_domain"]).size()
        .unstack(fill_value=0)
        .reindex(index=range(start_year, end_year + 1), columns=domains, fill_value=0)
    )
    long = wide.stack().reset_index(name="count")
    long.columns = ["year", "mit_risk_domain", "count"]
    long["mit_risk_domain"] = pd.Categorical(long["mit_risk_domain"], categories=domains)
    expected = (end_year - start_year + 1) * len(domains)
    assert len(long) == expected, f"expected {expected} rows, got {len(long)}"
    return long

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Poisson regression of annual incident count on year, risk domain, and
# their interaction. A significant interaction (via LR test against the
# no-interaction model) rejects parallel category-specific trends.
year_domain_long = zero_filled_long(classified, *PRIMARY_WINDOW)

full_model = smf.glm(
    "count ~ C(year) * C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
reduced_model = smf.glm(
    "count ~ C(year) + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()

lr_stat = 2 * (full_model.llf - reduced_model.llf)
lr_df = full_model.df_model - reduced_model.df_model
lr_pvalue = chi2.sf(lr_stat, lr_df)

print(f"LR test for parallel trends: LR = {lr_stat:.1f}, df = {lr_df:.0f}, p = {lr_pvalue:.3g}")
print()

# Per-category annual growth rate: exp(year coefficient) from a separate
# log-linear Poisson fit of count ~ year within each domain.
print("Estimated annual growth multiplier by risk domain:")
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    m = smf.glm("count ~ year", data=sub, family=sm.families.Poisson()).fit()
    growth = np.exp(m.params["year"])
    print(f"  {d}: {growth:.3f} ({(growth - 1):+.1%} per year)")

### Linear-trend robustness specification (primary reported statistic)

The interaction test above uses a fully saturated `C(year) * C(mit_risk_domain)`
model, which throws a `PerfectSeparationWarning` from zero-count year-domain
cells and has df = 39. The paper instead reports a more parsimonious linear
year-trend interaction test (`t * C(mit_risk_domain)`, df = 6) as the primary
statistic. This cell adds that specification without altering the saturated
model above.

In [ ]:
import warnings

# Reuse year_domain_long from the cell above; add a numeric time index if
# not already present (t = year - year.min(), i.e. 0-7 for 2018-2025).
if "t" not in year_domain_long.columns:
    year_domain_long["t"] = year_domain_long["year"] - year_domain_long["year"].min()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    full_linear = smf.glm(
        "count ~ t * C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
    ).fit()
    reduced_linear = smf.glm(
        "count ~ t + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
    ).fit()

lr_linear = 2 * (full_linear.llf - reduced_linear.llf)
lr_linear_df = full_linear.df_model - reduced_linear.df_model
lr_linear_p = chi2.sf(lr_linear, lr_linear_df)

print(f"LR test for parallel trends (linear year trend): LR = {lr_linear:.1f}, "
      f"df = {lr_linear_df:.0f}, p = {lr_linear_p:.3g}")

separation_warnings = [w for w in caught if "eparation" in str(w.message)]
if separation_warnings:
    print("\nWARNING: PerfectSeparationWarning raised for the linear-trend model "
          "(unexpected -- investigate before reporting this statistic):")
    for w in separation_warnings:
        print(" ", w.category.__name__, "-", w.message)
else:
    print("No PerfectSeparationWarning raised for the linear-trend model (as expected).")

# Per-domain growth rate under the linear (t) spec, cross-checked against the
# `count ~ year` spec from the previous cell. exp(coef on `year`) and
# exp(coef on `t = year - min(year)`) should be numerically identical --
# only the intercept absorbs the shift -- so confirm rather than assume.
print("\nPer-domain growth multiplier, t-spec vs. year-spec:")
mismatches = []
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    m_year = smf.glm("count ~ year", data=sub, family=sm.families.Poisson()).fit()
    m_t = smf.glm("count ~ t", data=sub, family=sm.families.Poisson()).fit()
    growth_year = np.exp(m_year.params["year"])
    growth_t = np.exp(m_t.params["t"])
    match = np.isclose(growth_year, growth_t, rtol=1e-6)
    if not match:
        mismatches.append((d, growth_year, growth_t))
    print(f"  {d}: t-spec = {growth_t:.3f}, year-spec = {growth_year:.3f} {'OK' if match else 'MISMATCH'}")

if mismatches:
    print("\nWARNING: growth multipliers differ between the `t` and `year` specifications:")
    for d, gy, gt in mismatches:
        print(f"  {d}: year-spec={gy:.6f} vs t-spec={gt:.6f}")
else:
    print("\nConfirmed: growth multipliers are identical (as expected) between specs.")

In [ ]:
from scipy.stats import norm


def mann_kendall(series):
    """Original (non-seasonal) Mann-Kendall trend test.

    Returns (trend, S, p_value) where trend is 'increasing', 'decreasing',
    or 'no trend' at the 0.05 level.
    """
    x = np.asarray(series, dtype=float)
    n = len(x)
    s = sum(np.sign(x[j] - x[i]) for i in range(n - 1) for j in range(i + 1, n))

    # Variance of S (no tie correction needed: annual incident counts are
    # essentially continuous-valued for n >= ~8 non-degenerate years).
    var_s = n * (n - 1) * (2 * n + 5) / 18

    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p_value = 2 * (1 - norm.cdf(abs(z)))
    if p_value < 0.05:
        trend = "increasing" if s > 0 else "decreasing"
    else:
        trend = "no trend"
    return trend, s, p_value


# Trend test on each domain's full annual absolute-count series (2018-2025).
annual_counts = classified.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
annual_counts = annual_counts.reindex(
    index=range(PRIMARY_WINDOW[0], PRIMARY_WINDOW[1] + 1), columns=domains, fill_value=0
)

print("Mann-Kendall trend test (annual absolute incident count, 2018-2025):")
for d in domains:
    trend, s, p = mann_kendall(annual_counts[d])
    print(f"  {d}: {trend} (S = {s:.0f}, p = {p:.3g})")

Note: three of seven domains show a statistically significant trend at the 5%
level over 2018-2025 -- Malicious Actors & Misuse, Misinformation, and
Privacy & Security are all increasing; the remaining four (Discrimination
and Toxicity, AI System Safety and Failures, Human-Computer Interaction,
Socioeconomic and Environmental Harms) show no significant trend. This is
inconsistent with a paper draft that describes only two categories as
increasing, and should be reconciled before finalizing Section 4.1.

## Window Robustness: Sensitivity to Analysis Start Year

Re-runs the linear year-trend interaction test and the Malicious Actors &
Misuse Mann-Kendall trend test across three window start years (2016, 2018,
2020), all ending in 2025, to check whether the compositional finding is
sensitive to where the primary analysis window begins.

In [ ]:
START_YEARS = [2016, 2018, 2020]
END_YEAR = 2025

window_rows = []
mk_by_window = {}
mk_p_by_window = {}

for start_year in START_YEARS:
    window = df[
        df["year"].between(start_year, END_YEAR)
        & df["mit_risk_domain"].notna()
        & (df["mit_risk_domain"] != "")
        & df["mit_risk_domain"].isin(domains)
    ].copy()

    long_w = zero_filled_long(window, start_year, END_YEAR)
    long_w["t"] = long_w["year"] - start_year

    # Wide year x domain count grid for this window. Built via
    # size().unstack(fill_value=0) -- NOT pivot()+reindex(fill_value=0) --
    # because reindex's fill_value only fills newly introduced index/column
    # labels; it does not backfill the NaNs pivot() leaves for zero-count
    # cells that fall within an already-present year/domain. unstack's
    # fill_value correctly fills those reshape-introduced NaNs.
    counts_wide = (
        window.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
        .reindex(index=range(start_year, END_YEAR + 1), columns=domains, fill_value=0)
    )

    zero_cells = [(d, int(y)) for y in counts_wide.index for d in domains if counts_wide.loc[y, d] == 0]
    if zero_cells:
        print(f"[start_year={start_year}] zero-count year/domain cell(s) (n={len(zero_cells)}):")
        for d, y in zero_cells:
            print(f"    {y} - {d}")

    full_w = smf.glm(
        "count ~ t * C(mit_risk_domain)", data=long_w, family=sm.families.Poisson()
    ).fit()
    red_w = smf.glm(
        "count ~ t + C(mit_risk_domain)", data=long_w, family=sm.families.Poisson()
    ).fit()
    lr_w = 2 * (full_w.llf - red_w.llf)
    lr_w_df = full_w.df_model - red_w.df_model
    lr_w_p = chi2.sf(lr_w, lr_w_df)

    window_rows.append({
        "start_year": start_year, "end_year": END_YEAR,
        "n": len(window), "LR": round(lr_w, 1), "df": int(lr_w_df), "p_value": lr_w_p,
    })

    # Mann-Kendall on the Malicious Actors & Misuse annual absolute count
    # for this window (reuses mann_kendall() defined in the cell above).
    # Both the trend label and its p-value are cached per window (keyed by
    # start_year) so downstream cells -- e.g. the Figure 1 plot -- can pull
    # them live instead of recomputing or, worse, hardcoding them.
    mam_series = counts_wide[mam_domain]
    trend, s, p = mann_kendall(mam_series)
    mk_by_window[start_year] = trend
    mk_p_by_window[start_year] = p
    print(f"[start_year={start_year}] Malicious Actors & Misuse Mann-Kendall: "
          f"{trend} (S={s:.0f}, p={p:.3g})")

window_robustness = pd.DataFrame(window_rows)
print()
print(window_robustness.to_string(index=False))

stable = len(set(mk_by_window.values())) == 1 and next(iter(mk_by_window.values())) == "increasing"
print()
if stable:
    print("Malicious Actors & Misuse trend classification is stable "
          "('increasing') across all three start years.")
else:
    print(f"Malicious Actors & Misuse trend classification is NOT stable "
          f"across start years: {mk_by_window}")

In [ ]:
import os

import matplotlib.pyplot as plt

# Figure 1: annual share of Malicious Actors & Misuse under each window
# specification, against the seven other risk domains shown faintly for
# reference. Every number that appears on the figure (n, LR test p-value,
# MK trend/p-value) is pulled live from window_robustness / mk_by_window /
# mk_p_by_window -- none of it is hardcoded, so the figure cannot drift out
# of sync with a later re-run of the underlying models.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

window_shares = {}
max_share_seen = 0.0
mam_handle = None

for start_year, ax in zip(START_YEARS, axes):
    window = df[
        df["year"].between(start_year, END_YEAR)
        & df["mit_risk_domain"].notna()
        & (df["mit_risk_domain"] != "")
        & df["mit_risk_domain"].isin(domains)
    ]

    # Per-year share of THIS window's own classified total -- not sliced
    # from the primary-window Table 2 percentages -- so shares sum to 100%
    # within each year of this window and pre-2018 years (start_year=2016)
    # are covered too.
    counts = (
        window.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
        .reindex(index=range(start_year, END_YEAR + 1), columns=domains, fill_value=0)
    )
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    window_shares[start_year] = pct
    max_share_seen = max(max_share_seen, pct.to_numpy().max())

    for d in domains:
        if d == mam_domain:
            continue
        ax.plot(pct.index, pct[d], color="lightgray", linewidth=1, zorder=1)

    line, = ax.plot(
        pct.index, pct[mam_domain], color="#1f77b4", linewidth=2.5, marker="o",
        zorder=3, label="Malicious Actors & Misuse",
    )
    mam_handle = line

    row = window_robustness.loc[window_robustness["start_year"] == start_year].iloc[0]
    trend = mk_by_window[start_year]
    mk_p = mk_p_by_window[start_year]

    # row['n'] would print as e.g. "1218.0" here: extracting a single row
    # from a DataFrame with mixed int/float columns upcasts the whole row
    # to float64. Cast back to int so it matches the verification printout
    # below exactly (same significant figures, same rounding).
    ax.set_title(
        f"Window {start_year}-{END_YEAR} (n={int(row['n'])})\n"
        f"Poisson interaction p={row['p_value']:.1e}\n"
        f"MK trend: {trend} (p={mk_p:.3f})"
    )
    ax.set_xlabel("Year")

axes[0].set_ylabel("Share of classified incidents (%)")
axes[0].set_ylim(0, max_share_seen * 1.1)

fig.legend(handles=[mam_handle], labels=["Malicious Actors & Misuse"], loc="upper left")
fig.suptitle("Window robustness check: 'Malicious Actors & Misuse' share under alternative start years")
plt.tight_layout(rect=[0, 0, 1, 0.93])

figure1_path = os.path.join(BATCH_DIR, "figure1_window_robustness.png")
plt.savefig(figure1_path, dpi=300)
plt.show()
print(f"Saved {figure1_path}")

# Verification: the exact numbers that appear in the three panel titles,
# printed here so they can be eyeballed against the figure directly.
print()
print("Panel title values (cross-check against the figure):")
title_check = window_robustness.copy()
title_check["mk_trend"] = title_check["start_year"].map(mk_by_window)
title_check["mk_p"] = title_check["start_year"].map(mk_p_by_window)
for _, r in title_check.iterrows():
    print(f"  start_year={r['start_year']}: n={int(r['n'])}, "
          f"Poisson interaction p={r['p_value']:.1e}, "
          f"MK trend={r['mk_trend']} (p={r['mk_p']:.3f})")

## 3.5 Media-Volume Normalization Robustness Check (H1a)

Tests whether the H1a absolute-count findings survive normalization against
an index of AI-related media/news volume, addressing the objection that
rising AIID incident counts reflect rising media attention to AI rather than
rising underlying incidence.

In [ ]:
%pip install -q pytrends

import io
import time

import requests
from requests.adapters import HTTPAdapter, Retry

GDELT_QUERY = '("artificial intelligence" OR "machine learning")'


# ============================================================================
# ARCHIVED -- GDELT DOC 2.0 fetch. NOT called by the active path below.
#
# Kept for reference / in case GDELT's legacy search API comes back online,
# but not currently usable: GDELT's own blog (June 2026, "Using The New Web
# NGrams Dataset To Find Relevant Coverage") states their legacy search/API
# infrastructure -- the doc/doc endpoint this code calls -- is overloaded
# during a migration to a new Spanner-based backend, and explicitly asks
# researchers to stop using the search APIs for the time being. This was
# independently confirmed by persistent 429s from Colab, from a local Mac
# (a dedicated, non-shared IP), and from a sandboxed dev environment --
# ruling out shared-IP contention or our own request pacing as the cause.
# Google Trends (below) is the active replacement.
# ============================================================================

def _fetch_gdelt_range(query, start, end, session):
    """One GDELT DOC 2.0 timelinevol request for a single [start, end] span."""
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": "timelinevol",
        "startdatetime": start,
        "enddatetime": end,
        "format": "csv",
    }
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text))
    # Column names vary slightly across GDELT responses; take date + value.
    date_col = [c for c in d.columns if "date" in c.lower()][0]
    val_col = [c for c in d.columns if c != date_col][-1]
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col])
    d["year"] = d[date_col].dt.year
    return d.groupby("year")[val_col].mean()


def fetch_gdelt_annual(query=GDELT_QUERY, y0=PRIMARY_WINDOW[0], y1=PRIMARY_WINDOW[1],
                        inter_request_sleep=8):
    """ARCHIVED -- see note above. Annual mean of GDELT DOC 2.0 daily volume
    intensity.

    'timelinevol' returns volume as a PERCENTAGE of all monitored articles,
    so it is already normalized for growth in total news output -- it
    isolates AI's *share* of media attention rather than conflating it with
    the general expansion of GDELT's crawl.

    Fetched one year at a time (rather than one request spanning the whole
    window) with a pause between requests: GDELT's rate limiter is far more
    likely to 429 a single multi-year request from a shared/cloud IP
    (Colab included) than a sequence of smaller ones. (In practice this
    still fails as of June 2026 -- see the ARCHIVED note above.)
    """
    # GDELT can be slow/flaky from cloud IPs (Colab included); retry a few
    # times with backoff rather than hanging on one long-timeout attempt.
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(
        total=4, backoff_factor=3, status_forcelist=[429, 500, 502, 503, 504]
    )))

    chunks = []
    for i, year in enumerate(range(y0, y1 + 1)):
        if i > 0:
            time.sleep(inter_request_sleep)
        chunk = _fetch_gdelt_range(query, f"{year}0101000000", f"{year}1231235959", session)
        chunks.append(chunk)

    combined = pd.concat(chunks)
    return combined.reindex(range(y0, y1 + 1))


# ============================================================================
# ARCHIVED -- Google Trends (pytrends) fetch. NOT called by the active path
# below.
#
# Kept as a live-fetch alternative in case the Drive file below is ever
# unavailable or needs refreshing, but not used by default: once the real
# GDELT values were verified (see below), there was no remaining reason to
# depend on a live fetch -- of GDELT (archived above, overloaded per
# GDELT's own blog) or of Trends (its own rate limits, and a conceptually
# weaker proxy: search interest, not news-coverage volume).
# ============================================================================

def fetch_pytrends_annual(keyword="artificial intelligence",
                           y0=PRIMARY_WINDOW[0], y1=PRIMARY_WINDOW[1]):
    """ARCHIVED -- see note above. Annual mean Google Trends search-interest
    index for `keyword`."""
    from pytrends.request import TrendReq

    pytrends = TrendReq(hl="en-US", tz=0)
    pytrends.build_payload([keyword], timeframe=f"{y0}-01-01 {y1}-12-31")
    monthly = pytrends.interest_over_time()
    if monthly.empty:
        raise RuntimeError(f"Google Trends returned no data for '{keyword}'.")
    annual = monthly[keyword].groupby(monthly.index.year).mean()
    return annual.reindex(range(y0, y1 + 1))


# ============================================================================
# ACTIVE -- load the verified GDELT index from a CSV in Google Drive. This
# is what `media` below uses.
#
# `module1_gdelt_media_index_verified.csv` holds real GDELT DOC 2.0
# timelinevol values, cross-validated across two independent successful
# fetches (this notebook's own GDELT cell in a prior session, and a
# standalone script run on a local Mac) agreeing to 4+ decimal places on
# every overlapping year -- strong evidence they are genuine, correct
# values. Reading a static, already-verified file sidesteps both GDELT's
# stated API overload and Google Trends' own rate limits entirely, rather
# than depending on either live fetch working on any given run.
# ============================================================================

GDELT_MEDIA_INDEX_FILENAME = "module1_gdelt_media_index_verified.csv"


def load_verified_media_index(filename=GDELT_MEDIA_INDEX_FILENAME,
                                y0=PRIMARY_WINDOW[0], y1=PRIMARY_WINDOW[1]):
    """Loads the verified annual GDELT media-volume index from BATCH_DIR."""
    path = os.path.join(BATCH_DIR, filename)
    d = pd.read_csv(path)
    d = d.set_index("year")
    value_col = [c for c in d.columns if c != "year"][0]
    return d[value_col].reindex(range(y0, y1 + 1))


# FALLBACK: if the Drive CSV above is ever missing or malformed, paste an
# annual series here by hand (e.g. from the Stanford AI Index media-
# attention data) and set USE_FALLBACK = True. Values need only be on a
# consistent relative scale -- the offset is scale-invariant up to an
# intercept shift. (Currently holds the same verified GDELT values as the
# Drive CSV, as a last-resort copy.)
USE_FALLBACK = False
FALLBACK_INDEX = {
    2018: 0.4489,
    2019: 0.4779,
    2020: 0.4110,
    2021: 0.4478,
    2022: 0.4479,
    2023: 1.1107,
    2024: 1.2958,
    2025: 1.7165,
}

# NOTE: raise RuntimeError, not SystemExit, below -- SystemExit inside a
# Jupyter/Colab cell breaks IPython's own traceback formatter (an "Internal
# Python error in the inspect module" crash unrelated to the real cause).
if USE_FALLBACK:
    media = pd.Series(FALLBACK_INDEX, name="media").astype(float)
else:
    try:
        media = load_verified_media_index()
        media.name = "media"
    except Exception as e:
        raise RuntimeError(
            f"Loading {GDELT_MEDIA_INDEX_FILENAME} from {BATCH_DIR} failed ({e}). "
            "Set USE_FALLBACK=True and paste an annual series into FALLBACK_INDEX."
        ) from e

if media.isna().any():
    raise RuntimeError(f"Index has missing years:\n{media}")

print("AI news-volume index (annual):")
print(media.round(4).to_string())
print()

# High correlation with calendar year is expected -- it is why the index
# enters as a fixed-coefficient offset rather than an estimated covariate.
r_yt = np.corrcoef(media.index.values, media.values)[0, 1]
print(f"corr(index, calendar year) = {r_yt:.3f}")
print("  -> high correlation is expected; it is the reason the index enters")
print("     as a fixed-coefficient offset rather than an estimated covariate.")

In [ ]:
# Rescale index to mean 1 so normalized counts stay in incident-like units
# and remain directly comparable to the raw counts.
m = media / media.mean()

media_comparison = pd.DataFrame({
    "raw": year_domain_counts.sum(axis=1),
    "index": m.round(3),
    "normalized": (year_domain_counts.sum(axis=1) / m).round(1),
})
media_comparison

In [ ]:
# Per-category trend (H1a): annual growth factor exp(beta), raw vs. media-
# normalized (offset Poisson: log(m) enters as a fixed-coefficient offset).
year_domain_long["t"] = year_domain_long["year"] - PRIMARY_WINDOW[0]
year_domain_long["log_m"] = np.log(year_domain_long["year"].map(m).to_numpy())

print(f"{'Risk domain':<50}{'raw':>10}{'normalized':>13}")
print("-" * 73)

media_trend_rows = []
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    raw = smf.glm("count ~ t", data=sub, family=sm.families.Poisson()).fit()
    adj = smf.glm("count ~ t", data=sub, family=sm.families.Poisson(), offset=sub["log_m"]).fit()

    def fmt(res):
        b = np.exp(res.params["t"])
        p = res.pvalues["t"]
        star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
        return f"{b:.3f}{star}"

    print(f"{d[:48]:<50}{fmt(raw):>10}{fmt(adj):>13}")
    media_trend_rows.append({
        "domain": d,
        "growth_raw": np.exp(raw.params["t"]), "p_raw": raw.pvalues["t"],
        "growth_norm": np.exp(adj.params["t"]), "p_norm": adj.pvalues["t"],
    })

media_trend = pd.DataFrame(media_trend_rows)
print("\n*** p<.001  ** p<.01  * p<.05")
print("Growth factor > 1 = increasing. Normalization shrinks all categories")
print("toward 1 because the common media trend is divided out.")

In [ ]:
# Year x category interaction (H1), raw vs. media-normalized. Uses a
# continuous linear trend (t) rather than the saturated C(year) model above,
# since the offset comparison requires a smooth exposure/trend specification.
full_raw = smf.glm(
    "count ~ t*C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
red_raw = smf.glm(
    "count ~ t + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
lr_raw = 2 * (full_raw.llf - red_raw.llf)
lr_raw_df = full_raw.df_model - red_raw.df_model

full_adj = smf.glm(
    "count ~ t*C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson(),
    offset=year_domain_long["log_m"],
).fit()
red_adj = smf.glm(
    "count ~ t + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson(),
    offset=year_domain_long["log_m"],
).fit()
lr_adj = 2 * (full_adj.llf - red_adj.llf)
lr_adj_df = full_adj.df_model - red_adj.df_model

print(f"  raw:        LR = {lr_raw:7.1f}, df = {lr_raw_df:.0f}, p = {chi2.sf(lr_raw, lr_raw_df):.2e}")
print(f"  normalized: LR = {lr_adj:7.1f}, df = {lr_adj_df:.0f}, p = {chi2.sf(lr_adj, lr_adj_df):.2e}")
print("""
These are near-identical by construction. A year-level multiplicative
factor scales every category identically within a year, so it cannot
create or destroy DIFFERENTIAL trends across categories -- and shares are
exactly invariant to it. With year fixed effects the offset is absorbed
completely and the statistic is unchanged to machine precision.

Report this as an analytic point, not an empirical discovery: the
compositional finding (H1) is immune to reporting-intensity drift by
construction. The normalization check earns its place against H1a, where
the claim concerns absolute levels.
""")

In [ ]:
# Mann-Kendall on raw vs. media-normalized annual counts (direct H1a check).
# Reuses the mann_kendall() function defined above rather than a separate
# pymannkendall dependency -- same test (original, non-seasonal MK).
print(f"{'Risk domain':<50}{'raw':>18}{'normalized':>20}")
print("-" * 88)

media_mk_rows = []
for d in domains:
    raw_series = annual_counts[d].astype(float)
    norm_series = annual_counts[d] / m.reindex(annual_counts.index)

    trend_raw, _, p_raw = mann_kendall(raw_series)
    trend_norm, _, p_norm = mann_kendall(norm_series)

    print(f"{d[:48]:<50}{trend_raw + ' ' + format(p_raw, '.3f'):>18}"
          f"{trend_norm + ' ' + format(p_norm, '.3f'):>20}")
    media_mk_rows.append({
        "domain": d, "trend_raw": trend_raw, "p_raw": p_raw,
        "trend_norm": trend_norm, "p_norm": p_norm,
    })

media_mk = pd.DataFrame(media_mk_rows)

print("""
n = 8 annual observations, so Mann-Kendall power is low; treat a shift from
'increasing' to 'no trend' as weak evidence rather than a refutation.

INTERPRETATION. The normalized series is a CONSERVATIVE LOWER BOUND, not a
preferred estimate. Media coverage of AI harm rose partly *because* AI harm
rose, so dividing by coverage removes real signal alongside reporting bias
(a bad-control problem). Report raw and normalized side by side and say so
explicitly. If misuse still trends up after normalization, that is a strong
result. If it flattens, that is ambiguous -- not a refutation of H1a.
""")

In [ ]:
import os

media_check_output = media_trend.merge(media_mk, on="domain")
media_check_output.to_csv(os.path.join(BATCH_DIR, "module1_media_normalization.csv"), index=False)
media_comparison.to_csv(os.path.join(BATCH_DIR, "module1_media_index.csv"))
print(f"Written module1_media_normalization.csv and module1_media_index.csv to {BATCH_DIR}")

# 4. Module 2 --- Correspondence with Postulated Threat Catalogues (H2)

Loads the pre-computed per-threat classification crosswalk (Claude / GPT /
Manual coders against the 55-threat catalogue) and its aggregate per-domain
distributions, already produced outside this notebook. Copy the three
`module2_*.csv` files (see `data/` in the repo) into `BATCH_DIR` in Drive
before running this section.

In [ ]:
MODULE2_AGG_CSV = os.path.join(BATCH_DIR, "module2_aggregate_distributions.csv")
MODULE2_CROSSWALK_CSV = os.path.join(BATCH_DIR, "module2_crosswalk_raw_per_threat.csv")
MODULE2_PAIRWISE_CSV = os.path.join(BATCH_DIR, "module2_pairwise_statistics.csv")

module2_agg = pd.read_csv(MODULE2_AGG_CSV)
module2_crosswalk = pd.read_csv(MODULE2_CROSSWALK_CSV)
module2_pairwise = pd.read_csv(MODULE2_PAIRWISE_CSV)

print(f"Aggregate distributions: {module2_agg.shape[0]} domains")
print(module2_agg.to_string(index=False))
print()
print(f"Per-threat crosswalk: {module2_crosswalk.shape[0]} threats")
print()
print("Pairwise statistics (as previously reported):")
print(module2_pairwise.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

SHORT_LABEL = {
    "Malicious Actors & Misuse": "Misuse",
    "AI System Safety, Failures, and Limitations": "Safety",
    "Discrimination and Toxicity": "Discrim.",
    "Misinformation": "Misinfo",
    "Privacy & Security": "Privacy",
    "Human-Computer Interaction": "HCI",
    "Socioeconomic & Environmental Harms": "Socioec.",
}

coders = [("claude_pct", "Claude"), ("gpt_pct", "GPT"), ("manual_pct", "Manual")]

# Single shared axis limit computed ONCE, before the panel loop. With
# sharex=True/sharey=True, a per-panel set_xlim/set_ylim call silently
# overrides every other panel's limits (whichever panel's call runs last
# wins), which would invisibly clip points from panels with a smaller data
# range (e.g. Manual tops out near 49% while Claude/GPT top out near
# 64-66% for Privacy & Security) with no error or warning.
all_values = pd.concat([module2_agg[col] for col, _ in coders] + [module2_agg["realized_pct"]])
lim = 1.15 * all_values.max()

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

pearson_computed = {}

for (col, name), ax in zip(coders, axes):
    x = module2_agg[col].to_numpy()
    y = module2_agg["realized_pct"].to_numpy()
    r, _ = pearsonr(x, y)
    pearson_computed[name] = r

    ax.scatter(x, y, color="#2a78d6", s=90, edgecolor="white", zorder=3)
    for xi, yi, label in zip(x, y, module2_agg["mit_domain_label"]):
        ax.annotate(
            SHORT_LABEL.get(label, label), (xi, yi),
            textcoords="offset points", xytext=(6, 4), fontsize=9,
        )

    ax.plot([0, lim], [0, lim], color="lightgray", linestyle="--", linewidth=1, zorder=1)

    slope, intercept = np.polyfit(x, y, 1)
    fit_x = np.array([0, lim])
    ax.plot(fit_x, slope * fit_x + intercept, color="orange", linewidth=2, zorder=2)

    ax.set_title(f"{name}\nPearson r = {r:.3f}")
    ax.set_xlabel("Catalogue share (%)")
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)

axes[0].set_ylabel("Realized incident share (%)")

# Verify all 7 points are visible in all 3 panels before saving.
for col, name in coders:
    exceeds = ((module2_agg[col] > lim) | (module2_agg["realized_pct"] > lim)).any()
    if exceeds:
        raise RuntimeError(f"{name}: a point exceeds the shared axis limit ({lim:.1f}) -- fix before saving.")
print(f"Shared axis limit: {lim:.1f}. Confirmed all 7 points visible in all 3 panels.")

fig.suptitle(
    "Catalogue Share vs. Realized Share, by MIT Harm Domain\n"
    "(gray dashed = perfect correspondence; orange = actual fit)"
)
plt.tight_layout(rect=[0, 0, 1, 0.90])

figure_scatter_path = "/content/drive/MyDrive/phd/P4/Data/figure_module2_catalogue_vs_realized.png"
plt.savefig(figure_scatter_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {figure_scatter_path}")

# Verification: reload the saved PNG and check its dimensions, and compare
# the computed Pearson r values against those already reported in the
# paper draft (per module2_pairwise_statistics.csv) to confirm the
# figure's titles match.
from PIL import Image

with Image.open(figure_scatter_path) as img:
    print(f"\nSaved figure dimensions: {img.size[0]} x {img.size[1]} px")

print()
print("Pearson r: computed here vs. paper draft values")
paper_r = {"Claude": -0.129, "GPT": -0.120, "Manual": 0.046}
for name in ["Claude", "GPT", "Manual"]:
    computed = pearson_computed[name]
    reported = paper_r[name]
    match = "MATCH" if abs(computed - reported) < 0.001 else "DIFFERS"
    print(f"  {name}: computed = {computed:.3f}, paper draft = {reported:.3f} [{match}]")

In [ ]:
gap_df = module2_agg.copy()
gap_df["short_label"] = gap_df["mit_domain_label"].map(lambda l: SHORT_LABEL.get(l, l))
gap_df["gap_claude"] = gap_df["realized_pct"] - gap_df["claude_pct"]
gap_df["gap_gpt"] = gap_df["realized_pct"] - gap_df["gpt_pct"]
gap_df["gap_manual"] = gap_df["realized_pct"] - gap_df["manual_pct"]
gap_df["gap_avg"] = gap_df[["gap_claude", "gap_gpt", "gap_manual"]].mean(axis=1)

# Ascending sort + matplotlib's default barh ordering (index 0 at the
# bottom) reads exactly as required: most-overrepresented-in-catalogue
# (most negative) at the bottom, most-underrepresented (most positive) at
# the top.
gap_sorted = gap_df.sort_values("gap_avg", ascending=True).reset_index(drop=True)

GAP_COLORS = {"Claude": "#888780", "GPT": "#d3d1c7", "Manual": "#eda100"}

fig, ax = plt.subplots(figsize=(9, 6))

y_pos = np.arange(len(gap_sorted))
h = 0.25

ax.barh(y_pos - h, gap_sorted["gap_claude"], height=h, color=GAP_COLORS["Claude"], label="Claude")
ax.barh(y_pos, gap_sorted["gap_gpt"], height=h, color=GAP_COLORS["GPT"], label="GPT")
ax.barh(y_pos + h, gap_sorted["gap_manual"], height=h, color=GAP_COLORS["Manual"], label="Manual")

ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(gap_sorted["short_label"])
ax.set_xlabel("Realized % − Catalogue % (percentage points)")
ax.set_title(
    "Where the Catalogue Misses Reality, by MIT Harm Domain\n"
    "(positive = underrepresented in catalogue; negative = overrepresented)"
)
ax.legend()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()

figure_gap_path = "/content/drive/MyDrive/phd/P4/Data/figure_module2_catalogue_gap_barchart.png"
plt.savefig(figure_gap_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {figure_gap_path}")

print()
print("Sorted gap table (percentage points, realized - catalogue):")
print(
    gap_sorted[["short_label", "gap_claude", "gap_gpt", "gap_manual"]]
    .round(1).to_string(index=False)
)

# 5. Module 3 --- Documented Legal and Financial Outcomes (Original Data)

Descriptive analysis of the hand-coded outcome sheet: for a stratified
sample of 150 incidents (drawn across risk domain and year), a systematic
search for documented legal or financial outcomes (litigation, settlements,
regulatory enforcement, quantified fines). Every number below is cross-checked
against the paper draft's Section 4.3 in Step 3.

In [ ]:
# Step 1: Load and validate
MODULE3_CSV = os.path.join(BATCH_DIR, "module3_coding_outcome_sheet.csv")

# Hand-coded outcome sheet, not the main AIID export -- read with
# dtype=str, keep_default_na=False (this project's established convention)
# so sample_order/incident_id aren't corrupted by integer/NaN coercion.
module3 = pd.read_csv(MODULE3_CSV, dtype=str, keep_default_na=False)

print(f"Loaded {len(module3)} rows, {len(module3.columns)} columns")
expected_n = 160
if len(module3) != expected_n:
    print(f"WARNING: expected {expected_n} rows, got {len(module3)}")

# Base sampled incidents (purely numeric sample_order) vs. extra
# multi-outcome rows (alphabetic suffix, e.g. "44b", "44c").
is_extra = module3["sample_order"].str.contains(r"[a-z]$", regex=True)
base = module3[~is_extra].copy()
extra = module3[is_extra].copy()

print(f"Base sampled incidents: {len(base)}")
print(f"Extra multi-outcome rows: {len(extra)}")

# Check 1: exactly 150 unique base incidents, no gaps in sample_order 1-150.
base_orders = sorted(base["sample_order"].astype(int))
expected_orders = list(range(1, 151))
check1_pass = (
    base["sample_order"].nunique() == 150
    and len(base) == 150
    and base_orders == expected_orders
)
print()
print(f"CHECK 1 (150 unique base incidents, sample_order 1-150 with no gaps): "
      f"{'PASS' if check1_pass else 'FAIL'}")
if not check1_pass:
    missing = sorted(set(expected_orders) - set(base_orders))
    dupes = base["sample_order"][base["sample_order"].duplicated()].tolist()
    print(f"  n unique = {base['sample_order'].nunique()}, n rows = {len(base)}")
    if missing:
        print(f"  missing sample_order values: {missing}")
    if dupes:
        print(f"  duplicated sample_order values: {dupes}")

# Check 2: no blank coder values (every row actually coded).
blank_coder = module3["coder"] == ""
check2_pass = not blank_coder.any()
print(f"CHECK 2 (no blank 'coder' values): {'PASS' if check2_pass else 'FAIL'}")
if not check2_pass:
    print(f"  {blank_coder.sum()} row(s) with blank coder:")
    print(module3.loc[blank_coder, ["sample_order", "incident_id", "title"]].to_string(index=False))

In [ ]:
# Step 2: Core descriptive statistics
from scipy.stats import norm

N_BASE = len(base)  # should be 150

# 1. Headline hit rate: base incidents with >=1 outcome_type != "none_found".
hit_mask = base["outcome_type"] != "none_found"
n_hits = int(hit_mask.sum())
hit_rate_pct = n_hits / N_BASE * 100
print(f"1. Headline hit rate: {n_hits} / {N_BASE} = {hit_rate_pct:.1f}%")

# 2. Outcome type breakdown -- ALL outcome-bearing rows (base + extra),
# since extra rows are additional outcome instances for multi-outcome
# incidents, not a separate population.
outcome_rows = module3[module3["outcome_type"] != "none_found"]
outcome_counts = outcome_rows["outcome_type"].value_counts()
print()
print("2. Outcome type breakdown (excluding none_found):")
print(outcome_counts.to_string())
print(f"   Total instances: {outcome_counts.sum()}")

# 3. 95% CI on the hit rate: Wald and Wilson.
p_hat = n_hits / N_BASE
z = norm.ppf(0.975)
wald_margin = z * np.sqrt(p_hat * (1 - p_hat) / N_BASE)
wald_lo, wald_hi = p_hat - wald_margin, p_hat + wald_margin

try:
    from statsmodels.stats.proportion import proportion_confint
    wilson_lo, wilson_hi = proportion_confint(n_hits, N_BASE, method="wilson")
except ImportError:
    denom = 1 + z**2 / N_BASE
    center = (p_hat + z**2 / (2 * N_BASE)) / denom
    margin = (z * np.sqrt(p_hat * (1 - p_hat) / N_BASE + z**2 / (4 * N_BASE**2))) / denom
    wilson_lo, wilson_hi = center - margin, center + margin

print()
print(f"3. 95% CI on hit rate ({hit_rate_pct:.1f}%, n={N_BASE}):")
print(f"   Wald:   [{wald_lo:.1%}, {wald_hi:.1%}]")
print(f"   Wilson: [{wilson_lo:.1%}, {wilson_hi:.1%}]")

# 4. Hit rate by MIT risk domain.
domain_summary = (
    base.assign(has_outcome=hit_mask)
    .groupby("mit_risk_domain")
    .agg(incidents_sampled=("sample_order", "count"), incidents_with_outcome=("has_outcome", "sum"))
    .reset_index()
)
domain_summary["hit_rate_pct"] = (
    domain_summary["incidents_with_outcome"] / domain_summary["incidents_sampled"] * 100
).round(1)
print()
print("4. Hit rate by MIT risk domain:")
print(domain_summary.to_string(index=False))

# 5. Line of business breakdown among outcome rows.
lob_counts = outcome_rows["line_of_business"].value_counts(dropna=False)
print()
print("5. Line of business breakdown (outcome rows only; blank = no clear assignment):")
print(lob_counts.to_string())

# 6. Disclosed amounts.
has_amount = module3["amount_usd"].str.strip() != ""
amounts = module3[has_amount].copy()
amounts["amount_usd_float"] = (
    amounts["amount_usd"].str.replace(r"[$,]", "", regex=True).astype(float)
)
amounts_sorted = amounts.sort_values("amount_usd_float")

print()
print(f"6. Disclosed amounts: n = {len(amounts_sorted)}")
if len(amounts_sorted):
    print(f"   Min: ${amounts_sorted['amount_usd_float'].min():,.0f}")
    print(f"   Max: ${amounts_sorted['amount_usd_float'].max():,.0f}")
print()
print(amounts_sorted[["incident_id", "title", "outcome_type", "amount_usd_float"]].to_string(index=False))

In [ ]:
# Step 3: Cross-check against the paper
print("Step 3: Cross-check against paper draft values (Section 4.3)")
print("=" * 70)

check_results = []
mismatched_labels = []


def check(label, computed, paper, tol=1e-9):
    """Print MATCH/MISMATCH for a single computed-vs-paper value.

    A mismatch is flagged explicitly rather than silently accepted --
    resolving which side is right is for a human, not this notebook.
    """
    ok = computed is not None and abs(computed - paper) <= tol
    line = f"  {label}: notebook = {computed}, paper = {paper}"
    if ok:
        print(line + "  [MATCH]")
    else:
        print(line + f"  [MISMATCH: paper states {paper}, notebook computes {computed}]")
    check_results.append(ok)
    if not ok:
        mismatched_labels.append(label)
    return ok


check("Headline hit rate (n)", n_hits, 33)
check("Headline hit rate (%)", round(hit_rate_pct, 1), 22.0)
check("Outcome type: litigation", int(outcome_counts.get("litigation", 0)), 14)
check("Outcome type: criminal_prosecution", int(outcome_counts.get("criminal_prosecution", 0)), 9)
check("Outcome type: fine", int(outcome_counts.get("fine", 0)), 8)
check("Outcome type: settlement", int(outcome_counts.get("settlement", 0)), 6)
check("Outcome type: regulatory_enforcement", int(outcome_counts.get("regulatory_enforcement", 0)), 6)
check("Outcome type: total instances", int(outcome_counts.sum()), 43)
# Wilson CI is cited in the paper as "approximately 16%-29%" -- compared
# to the nearest percentage point rather than requiring exact equality.
check("Wilson CI lower bound (pp)", round(wilson_lo * 100), 16, tol=1)
check("Wilson CI upper bound (pp)", round(wilson_hi * 100), 29, tol=1)
check("Disclosed amounts (n)", len(amounts_sorted), 17)
check(
    "Disclosed amounts (min $)",
    amounts_sorted["amount_usd_float"].min() if len(amounts_sorted) else None,
    1874,
)
check(
    "Disclosed amounts (max $)",
    amounts_sorted["amount_usd_float"].max() if len(amounts_sorted) else None,
    140_000_000,
)

# Domain hit rates: match the paper's short domain names to this dataset's
# full "N. <name>" mit_risk_domain labels (same MIT taxonomy as Module 1).
DOMAIN_PAPER = {
    "1. Discrimination and Toxicity": ("Discrimination", 2, 19),
    "2. Privacy & Security": ("Privacy", 4, 10),
    "3. Misinformation": ("Misinformation", 5, 22),
    "4. Malicious Actors & Misuse": ("Misuse", 11, 61),
    "5. Human-Computer Interaction": ("HCI", 2, 8),
    "6. Socioeconomic & Environmental Harms": ("Socioeconomic", 3, 7),
    "7. AI system safety, failures, and limitations": ("Safety", 6, 23),
}
domain_summary_idx = domain_summary.set_index("mit_risk_domain")
for full_label, (short, paper_hits, paper_total) in DOMAIN_PAPER.items():
    if full_label in domain_summary_idx.index:
        computed_hits = int(domain_summary_idx.loc[full_label, "incidents_with_outcome"])
        computed_total = int(domain_summary_idx.loc[full_label, "incidents_sampled"])
    else:
        computed_hits, computed_total = None, None
        print(f"  NOTE: domain label '{full_label}' not found in this dataset's mit_risk_domain values.")
    check(f"Domain hit rate: {short} (hits)", computed_hits, paper_hits)
    check(f"Domain hit rate: {short} (total)", computed_total, paper_total)

# Line of business (only the five categories the paper gives explicit
# counts for; the residual "no clear assignment"/"other" rows are reported
# in Step 2 #5 above without a specific paper target to check against).
LOB_PAPER = {
    "media_liability": 7, "cyber": 7, "E&O_professional": 6,
    "product_liability": 4, "GL": 3,
}
for lob, paper_n in LOB_PAPER.items():
    computed_n = int(lob_counts.get(lob, 0))
    check(f"Line of business: {lob}", computed_n, paper_n)

all_pass = all(check_results)
print()
if all_pass:
    print("All paper-stated numbers in Section 4.3 match this notebook's computation.")
else:
    print(f"{len(mismatched_labels)} mismatch(es) found: {mismatched_labels}")

In [ ]:
# Step 4: Save outputs
domain_out = domain_summary[["mit_risk_domain", "incidents_sampled", "incidents_with_outcome", "hit_rate_pct"]]
domain_out_path = os.path.join(BATCH_DIR, "module3_summary_by_domain.csv")
domain_out.to_csv(domain_out_path, index=False)

amounts_out = amounts_sorted[["incident_id", "title", "outcome_type", "amount_usd_float"]].rename(
    columns={"amount_usd_float": "amount_usd"}
)
amounts_out_path = os.path.join(BATCH_DIR, "module3_disclosed_amounts.csv")
amounts_out.to_csv(amounts_out_path, index=False)

print(f"Saved {domain_out_path}")
print(f"Saved {amounts_out_path}")
print()
if all_pass:
    print("SUMMARY: every number in the paper's Section 4.3 is traceable to this notebook and matches.")
else:
    print("SUMMARY: NOT all numbers match. Mismatches found in:")
    for label in mismatched_labels:
        print(f"  - {label}")

### Module 3b: build coding template

In [ ]:
# Module 3b, Cell A: build coding template.
#
# Never overwrites module3_coding_outcome_sheet.csv -- only reads it. All
# output goes to a NEW file, module3_coding_outcome_sheet_coded.csv.
MODULE3_PRECODED_CSV = os.path.join(BATCH_DIR, "module3_disclosed_amounts_precoded.csv")
MODULE3_CODED_CSV = os.path.join(BATCH_DIR, "module3_coding_outcome_sheet_coded.csv")

# Step 1: inspect the outcome sheet and identify outcome rows.
# Reuses `module3` (loaded dtype=str, keep_default_na=False in Module 3 Step 1
# above) rather than re-reading the file -- same convention, same object.
print(f"module3_coding_outcome_sheet.csv columns: {list(module3.columns)}")

OUTCOME_COL = "outcome_type" if "outcome_type" in module3.columns else None
if OUTCOME_COL is None:
    raise ValueError("Could not find an outcome_type column (or equivalent) marking outcomes.")

is_outcome_row = (module3[OUTCOME_COL] != "") & (module3[OUTCOME_COL] != "none_found")
outcome_rows_raw = module3[is_outcome_row]
n_outcome_rows = len(outcome_rows_raw)
n_outcome_incidents = outcome_rows_raw["incident_id"].nunique()
print(f"\nOutcome rows: {n_outcome_rows} (expected 43)")
print(f"Distinct incidents with an outcome: {n_outcome_incidents} (expected 33)")
if n_outcome_rows != 43:
    print(f"WARNING: expected 43 outcome rows, got {n_outcome_rows}")
if n_outcome_incidents != 33:
    print(f"WARNING: expected 33 distinct incidents with an outcome, got {n_outcome_incidents}")

# Step 2: skip entirely if the coded file already exists -- never regenerate
# over hand-coding work already done.
if os.path.exists(MODULE3_CODED_CSV):
    print("\ncoded file exists, skipping template")
else:
    # Step 3: copy the outcome sheet, add the new (blank) columns.
    NEW_COLS = [
        "respondent_role", "respondent_name", "payment_nature", "amount_status",
        "outcome_type_revised", "line_of_business_suggested", "respondent_note",
    ]
    coded = module3.copy()
    for col in NEW_COLS:
        coded[col] = ""

    precoded = pd.read_csv(MODULE3_PRECODED_CSV, dtype=str, keep_default_na=False)
    print(f"\nmodule3_disclosed_amounts_precoded.csv columns: {list(precoded.columns)}")

    def _parse_amount(series):
        """'$1,874' -> 1874.00, '' -> NaN, rounded to 2dp for matching."""
        return pd.to_numeric(
            series.str.strip().str.replace(r"[$,]", "", regex=True), errors="coerce"
        ).round(2)

    coded["_amount_key"] = _parse_amount(coded["amount_usd"])
    precoded["_amount_key"] = _parse_amount(precoded["amount_usd"])

    precoded_dup = precoded.duplicated(subset=["incident_id", "_amount_key"], keep=False)
    if precoded_dup.any():
        print("\nWARNING: duplicate (incident_id, amount_usd) keys within the precoded file itself:")
        print(precoded.loc[precoded_dup].to_string(index=False))

    # Step 4: match on (incident_id, amount_usd rounded to 2dp); must be
    # exactly 17 rows, one-to-one. If not, print the unmatched rows on both
    # sides and stop -- do not guess or partially proceed.
    merged = coded.merge(
        precoded, on=["incident_id", "_amount_key"], how="outer", suffixes=("", "_precoded"), indicator=True
    )
    matched = merged[merged["_merge"] == "both"]
    coded_only = merged[(merged["_merge"] == "left_only") & merged["_amount_key"].notna()]
    precoded_only = merged[merged["_merge"] == "right_only"]

    n_matches = len(matched)
    print(f"\nMatched {n_matches} row(s) on (incident_id, amount_usd rounded to 2dp) "
          f"(expected 17, one-to-one).")

    one_to_one_ok = n_matches == 17 and not matched.duplicated(subset=["incident_id", "_amount_key"]).any()

    if not one_to_one_ok:
        print("\nSTOP: match is not exactly 17 one-to-one rows.")
        if len(coded_only):
            print(f"\n{len(coded_only)} outcome-sheet row(s) with an amount but no precoded match:")
            print(coded_only[["incident_id", "title", "amount_usd"]].to_string(index=False))
        if len(precoded_only):
            print(f"\n{len(precoded_only)} precoded row(s) with no match in the outcome sheet:")
            print(precoded_only[["incident_id", "amount_usd"]].to_string(index=False))
        raise AssertionError(f"Expected exactly 17 one-to-one matches on (incident_id, amount_usd); got {n_matches}.")

    print("Exactly 17 one-to-one matches confirmed.")

    # Fill the new columns from whichever of them the precoded file actually
    # provides -- vectorized merge rather than a per-row loop.
    precoded_fill_cols = [c for c in NEW_COLS if c in precoded.columns]
    missing_fill_cols = [c for c in NEW_COLS if c not in precoded.columns]
    if missing_fill_cols:
        print(f"\nprecoded file has no column(s) {missing_fill_cols} -- left blank for matched rows")

    precoded_slim = precoded[["incident_id", "_amount_key"] + precoded_fill_cols].rename(
        columns={c: f"{c}__fill" for c in precoded_fill_cols}
    )
    coded = coded.merge(precoded_slim, on=["incident_id", "_amount_key"], how="left")
    for c in precoded_fill_cols:
        fillcol = f"{c}__fill"
        mask = coded[fillcol].notna() & (coded[fillcol] != "")
        coded.loc[mask, c] = coded.loc[mask, fillcol]
        coded = coded.drop(columns=[fillcol])

    # Step 5: outcome_type_revised defaults to outcome_type where not filled.
    needs_default = coded["outcome_type_revised"].astype(str).str.strip() == ""
    coded.loc[needs_default, "outcome_type_revised"] = coded.loc[needs_default, OUTCOME_COL]

    # Step 6: save (never touching module3_coding_outcome_sheet.csv itself)
    # and report what's left to hand-code.
    coded = coded.drop(columns=["_amount_key"])
    coded.to_csv(MODULE3_CODED_CSV, index=False)
    print(f"\nSaved {MODULE3_CODED_CSV}")

    coded_outcome_mask = (coded[OUTCOME_COL] != "") & (coded[OUTCOME_COL] != "none_found")
    needs_hand_coding = coded[
        coded_outcome_mask & ((coded["respondent_role"] == "") | (coded["payment_nature"] == ""))
    ]
    print(f"\nOutcome rows still needing hand-coding (blank respondent_role or payment_nature): "
          f"{len(needs_hand_coding)} (expected 26)")
    print(needs_hand_coding[["sample_order", "incident_id", "title", "outcome_type", "amount_usd"]].to_string(index=False))

### Module 3b: validation

In [ ]:
# Module 3b, Cell B: validation.
#
# Reads back module3_coding_outcome_sheet_coded.csv (not module3 in memory)
# so this reflects whatever hand-coding has actually been saved to Drive.
coded_v = pd.read_csv(MODULE3_CODED_CSV, dtype=str, keep_default_na=False)

OUTCOME_COL_V = "outcome_type" if "outcome_type" in coded_v.columns else None
if OUTCOME_COL_V is None:
    raise ValueError("Could not find an outcome_type column in the coded file.")

outcome_mask_v = (coded_v[OUTCOME_COL_V] != "") & (coded_v[OUTCOME_COL_V] != "none_found")
outcome_rows_v = coded_v[outcome_mask_v]

# Drop excluded outcome rows before any analysis. A missing column means no
# exclusions at all (not an error).
if "exclude_from_analysis" in coded_v.columns:
    excl_mask_v = outcome_rows_v["exclude_from_analysis"].str.strip().str.lower() == "yes"
    outcome_rows_v = outcome_rows_v[~excl_mask_v]

print(f"Validating {len(outcome_rows_v)} outcome row(s) in {MODULE3_CODED_CSV}")

VOCAB = {
    "respondent_role": {
        "developer", "developer_deployer", "deployer_firm", "deployer_public",
        "perpetrator", "user_individual", "intermediary", "other", "unknown",
    },
    "payment_nature": {"compensatory", "penalty", "criminal", "none", "unknown"},
    "amount_status": {"paid", "agreed", "ordered", "ordered_unpaid", "sought", "claimed"},
}

# 1. Vocabulary membership. Every blank or invalid value is printed with
# incident_id and flagged as a warning, but this never halts the run --
# that's what Cell A's "needs hand-coding" list and this cell together are
# for, not a hard gate.
print("\n=== Check 1: vocabulary membership (outcome rows) ===")
any_vocab_issue = False
for col, vocab in VOCAB.items():
    vals = outcome_rows_v[col]
    blanks = outcome_rows_v[vals == ""]
    invalid = outcome_rows_v[(vals != "") & ~vals.isin(vocab)]
    if len(blanks):
        any_vocab_issue = True
        print(f"\nWARNING: {len(blanks)} outcome row(s) with blank {col}:")
        print(blanks[["incident_id", "title"]].to_string(index=False))
    if len(invalid):
        any_vocab_issue = True
        print(f"\nWARNING: {len(invalid)} outcome row(s) with invalid {col} value:")
        print(invalid[["incident_id", "title", col]].to_string(index=False))
    if not len(blanks) and not len(invalid):
        print(f"  {col}: OK (no blanks or invalid values)")
if not any_vocab_issue:
    print("\nNo vocabulary issues found.")

# 2. Consistency checks. Printed, never auto-fixed -- resolving a
# consistency violation is for a human, same convention as the paper
# cross-checks earlier in this module.
print("\n=== Check 2: consistency (outcome rows) ===")

amt_v = pd.to_numeric(
    outcome_rows_v["amount_usd"].str.strip().str.replace(r"[$,]", "", regex=True), errors="coerce"
)
has_amt_v = amt_v.notna()

v1 = outcome_rows_v[has_amt_v & (outcome_rows_v["amount_status"] == "")]
print(f"\n  amount_usd present -> amount_status not blank: {len(v1)} violation(s)")
if len(v1):
    print(v1[["incident_id", "title", "amount_usd", "amount_status"]].to_string(index=False))

v2 = outcome_rows_v[
    outcome_rows_v["payment_nature"].isin(["compensatory", "penalty", "criminal"])
    & ~has_amt_v
    & (outcome_rows_v["amount_status"] != "")
]
print(f"\n  payment_nature in {{compensatory,penalty,criminal}} -> amount_usd present, "
      f"or amount_status blank: {len(v2)} violation(s)")
if len(v2):
    print(v2[["incident_id", "title", "payment_nature", "amount_usd", "amount_status"]].to_string(index=False))

v3 = outcome_rows_v[
    (outcome_rows_v["payment_nature"] == "criminal")
    & ~outcome_rows_v["respondent_role"].isin(["perpetrator", "user_individual"])
]
print(f"\n  payment_nature == 'criminal' -> respondent_role in {{perpetrator,user_individual}}: "
      f"{len(v3)} violation(s)")
if len(v3):
    print(v3[["incident_id", "title", "payment_nature", "respondent_role"]].to_string(index=False))

### Module 3b: respondent and payment analysis

In [ ]:
%pip install -q openpyxl

# Module 3b, Cell C: respondent and payment analysis.
# Uses outcome_type_revised throughout (not the original outcome_type).
coded_c = pd.read_csv(MODULE3_CODED_CSV, dtype=str, keep_default_na=False)

OUTCOME_COL_C = "outcome_type" if "outcome_type" in coded_c.columns else None
outcome_mask_c = (coded_c[OUTCOME_COL_C] != "") & (coded_c[OUTCOME_COL_C] != "none_found")
outcome_rows_c = coded_c[outcome_mask_c].copy()

# Drop excluded outcome rows before any analysis. A missing column means no
# exclusions at all (not an error). Printed here, at the top of Cell C, per
# instruction -- Cell B drops silently (no print) since it only validates.
if "exclude_from_analysis" in coded_c.columns:
    excl_mask_c = outcome_rows_c["exclude_from_analysis"].str.strip().str.lower() == "yes"
else:
    excl_mask_c = pd.Series(False, index=outcome_rows_c.index)
excluded_rows_c = outcome_rows_c[excl_mask_c]

print(f"Excluded outcome row(s) (exclude_from_analysis == 'yes'): {len(excluded_rows_c)}")
if len(excluded_rows_c):
    excl_cols = [c for c in ["sample_order", "incident_id", "exclude_reason"] if c in excluded_rows_c.columns]
    print(excluded_rows_c[excl_cols].to_string(index=False))

outcome_rows_c = outcome_rows_c[~excl_mask_c].copy()

outcome_rows_c["amount_usd_float"] = pd.to_numeric(
    outcome_rows_c["amount_usd"].str.strip().str.replace(r"[$,]", "", regex=True), errors="coerce"
)
VALID_STATUSES = {"paid", "agreed", "ordered", "ordered_unpaid"}
outcome_rows_c["is_valid_amount"] = outcome_rows_c["amount_status"].isin(VALID_STATUSES)

OUTCOMES_AS_OF = "2026-11"

# 1. Coverage.
n_outcome_rows_c = len(outcome_rows_c)
n_incidents_outcome = outcome_rows_c["incident_id"].nunique()
n_sampled_c = N_BASE  # reuses N_BASE (=150) from Module 3 Step 2 above
pct_of_sampled = n_incidents_outcome / n_sampled_c * 100 if n_sampled_c else float("nan")
print(f"1. Coverage (after exclusions): {n_outcome_rows_c} outcome rows, {n_incidents_outcome} "
      f"distinct incidents with >=1 remaining outcome, {n_incidents_outcome} / {n_sampled_c} sampled "
      f"({pct_of_sampled:.1f}%)")

# Hit rate by mit_risk_domain, after exclusions. Reuses `base` (the 150
# sampled incidents, dtype=str) from Module 3's own Step 1/2 cells above --
# not rebuilt, not modified.
incidents_with_remaining_outcome = set(outcome_rows_c["incident_id"])
base_final = base.copy()
base_final["has_outcome_final"] = base_final["incident_id"].isin(incidents_with_remaining_outcome)

domain_summary_final = (
    base_final.groupby("mit_risk_domain")
    .agg(incidents_sampled=("sample_order", "count"), incidents_with_outcome=("has_outcome_final", "sum"))
    .reset_index()
)
domain_summary_final["hit_rate_pct"] = (
    domain_summary_final["incidents_with_outcome"] / domain_summary_final["incidents_sampled"] * 100
).round(1)

print("\n   Hit rate by mit_risk_domain, after exclusions:")
print(domain_summary_final.to_string(index=False))

domain_final_path = os.path.join(BATCH_DIR, "module3_summary_by_domain_final.csv")
domain_summary_final.to_csv(domain_final_path, index=False)
print(f"   Saved {domain_final_path}")

# 2. respondent_role counts + incident-level role flags.
print("\n2. respondent_role counts (outcome rows):")
role_counts = outcome_rows_c["respondent_role"].value_counts(dropna=False)
print(role_counts.to_string())

DEVELOPER_ROLES = {"developer", "developer_deployer"}
DEPLOYER_ROLES = {"deployer_firm", "deployer_public", "developer_deployer"}
PERP_USER_ROLES = {"perpetrator", "user_individual"}
INTERMEDIARY_OTHER_ROLES = {"intermediary", "other"}


def _incident_level_flag(roles_set):
    g = outcome_rows_c.groupby("incident_id")["respondent_role"].apply(lambda roles: bool(set(roles) & roles_set))
    n = int(g.sum())
    pct = n / n_incidents_outcome * 100 if n_incidents_outcome else float("nan")
    return n, pct


print()
for label, roles_set in [
    ("developer (developer, developer_deployer)", DEVELOPER_ROLES),
    ("deployer (deployer_firm, deployer_public, developer_deployer)", DEPLOYER_ROLES),
    ("perpetrator or user_individual", PERP_USER_ROLES),
    ("intermediary or other", INTERMEDIARY_OTHER_ROLES),
]:
    n, pct = _incident_level_flag(roles_set)
    print(f"  incidents with >=1 {label}: {n} ({pct:.1f}%)")

# 3. Cross-tabs (outcome rows).
print("\n3. Cross-tabs (outcome rows):")
ct_role_outcome = pd.crosstab(outcome_rows_c["respondent_role"], outcome_rows_c["outcome_type_revised"])
print("\n  respondent_role x outcome_type_revised:")
print(ct_role_outcome.to_string())

ct_role_lob = pd.crosstab(outcome_rows_c["respondent_role"], outcome_rows_c["line_of_business"])
print("\n  respondent_role x line_of_business:")
print(ct_role_lob.to_string())

ct_nature_outcome = pd.crosstab(outcome_rows_c["payment_nature"], outcome_rows_c["outcome_type_revised"])
print("\n  payment_nature x outcome_type_revised:")
print(ct_nature_outcome.to_string())

# 4. line_of_business == "none"(-like).
lob_lower = outcome_rows_c["line_of_business"].str.lower()
none_mask = lob_lower.str.startswith("none")
print(f"\n4. line_of_business == 'none' (or starting with 'none'): {int(none_mask.sum())} outcome row(s)")
print("\n  by respondent_role:")
print(outcome_rows_c.loc[none_mask, "respondent_role"].value_counts().to_string())
print("\n  by payment_nature:")
print(outcome_rows_c.loc[none_mask, "payment_nature"].value_counts().to_string())

# 5. Disclosed amounts (VALID only), at three levels.
valid_rows = outcome_rows_c[outcome_rows_c["is_valid_amount"] & outcome_rows_c["amount_usd_float"].notna()]
print(f"\n5. Disclosed amounts (VALID only: amount_status in {sorted(VALID_STATUSES)})")

# (a) row level, by payment_nature
row_level = valid_rows.groupby("payment_nature")["amount_usd_float"].agg(n="count", total="sum", median="median")
row_level = row_level.reset_index()
total_all_rows = row_level["total"].sum()
row_level["share_of_total_pct"] = row_level["total"] / total_all_rows * 100 if total_all_rows else float("nan")
print("\n  (a) row level, by payment_nature:")
print(row_level.to_string(index=False))

# (b) incident level: sum within incident_id
incident_level = valid_rows.groupby("incident_id")["amount_usd_float"].sum()
print(f"\n  (b) incident level: n incidents = {len(incident_level)}, "
      f"total = ${incident_level.sum():,.0f}, median = ${incident_level.median():,.0f}")

# (c) respondent level: sum within (incident_id, respondent_name); a
# respondent's payment_nature/respondent_role are taken from its LARGEST
# single row (ties resolved by idxmax's first-match default).
resp_group = valid_rows.groupby(["incident_id", "respondent_name"], group_keys=False)
respondent_level = resp_group["amount_usd_float"].sum().reset_index(name="total_amount")


def _largest_row_value(g, col):
    return g.loc[g["amount_usd_float"].idxmax(), col]


resp_nature = resp_group.apply(lambda g: _largest_row_value(g, "payment_nature"), include_groups=False).reset_index(name="payment_nature")
resp_role = resp_group.apply(lambda g: _largest_row_value(g, "respondent_role"), include_groups=False).reset_index(name="respondent_role")
respondent_level = respondent_level.merge(resp_nature, on=["incident_id", "respondent_name"])
respondent_level = respondent_level.merge(resp_role, on=["incident_id", "respondent_name"])

print(f"\n  (c) respondent level: n respondents = {len(respondent_level)}, "
      f"total = ${respondent_level['total_amount'].sum():,.0f}, "
      f"median = ${respondent_level['total_amount'].median():,.0f}")
print("\n     by respondent_role:")
print(respondent_level.groupby("respondent_role")["total_amount"].agg(n="count", total="sum", median="median")
      .to_string())
print("\n     by payment_nature:")
print(respondent_level.groupby("payment_nature")["total_amount"].agg(n="count", total="sum", median="median")
      .to_string())

# Sought/claimed amounts -- reported separately, excluded from all of (a)-(c).
sought_claimed = outcome_rows_c[outcome_rows_c["amount_status"].isin(["sought", "claimed"])]
sc_amounts = sought_claimed["amount_usd_float"].dropna()
print(f"\n  Sought/claimed amounts (excluded from (a)-(c) above): "
      f"n = {len(sc_amounts)}, total = ${sc_amounts.sum():,.0f}")

# 6. Cover-limit check at respondent level.
COVER_LIMIT = 50_000_000
print(f"\n6. Cover-limit check at respondent level (<= ${COVER_LIMIT:,.0f}):")
within_limit = respondent_level["total_amount"] <= COVER_LIMIT
n_within = int(within_limit.sum())
pct_within = n_within / len(respondent_level) * 100 if len(respondent_level) else float("nan")
print(f"  Overall: {n_within} / {len(respondent_level)} ({pct_within:.1f}%) <= ${COVER_LIMIT:,.0f}")

print("\n  by payment_nature:")
for nature, g in respondent_level.groupby("payment_nature"):
    n_w = int((g["total_amount"] <= COVER_LIMIT).sum())
    pct_w = n_w / len(g) * 100 if len(g) else float("nan")
    print(f"    {nature}: {n_w} / {len(g)} ({pct_w:.1f}%)")

above_limit = respondent_level[respondent_level["total_amount"] > COVER_LIMIT]
print(f"\n  Respondents above ${COVER_LIMIT:,.0f}:")
print(above_limit[["incident_id", "respondent_name", "total_amount", "respondent_role", "payment_nature"]]
      .to_string(index=False))

# 7. Incidents with more than one outcome row.
print("\n7. Incidents with more than one outcome row:")
multi = outcome_rows_c.groupby("incident_id").agg(
    n_outcomes=("sample_order", "count"),
    distinct_respondent_names=("respondent_name", lambda s: sorted(set(x for x in s if x))),
    respondent_roles=("respondent_role", lambda s: sorted(set(x for x in s if x))),
    payment_natures=("payment_nature", lambda s: sorted(set(x for x in s if x))),
).reset_index()
multi = multi[multi["n_outcomes"] > 1]
print(multi.to_string(index=False))

# 8. Paper table.
ROLE_GROUPS = {
    "developer": ["developer"],
    "developer_deployer": ["developer_deployer"],
    "deployer_firm": ["deployer_firm"],
    "deployer_public": ["deployer_public"],
    "perpetrator & user_individual": ["perpetrator", "user_individual"],
    "intermediary & other": ["intermediary", "other"],
}

paper_rows = []
for group_label, roles in ROLE_GROUPS.items():
    rows_g = outcome_rows_c[outcome_rows_c["respondent_role"].isin(roles)]
    n_incidents_g = rows_g["incident_id"].nunique()
    n_outcomes_g = len(rows_g)
    n_with_amount_g = int(rows_g["amount_usd_float"].notna().sum())

    valid_rows_g = valid_rows[valid_rows["respondent_role"].isin(roles)]
    comp_usd = valid_rows_g.loc[valid_rows_g["payment_nature"] == "compensatory", "amount_usd_float"].sum()
    pen_usd = valid_rows_g.loc[valid_rows_g["payment_nature"] == "penalty", "amount_usd_float"].sum()
    crim_usd = valid_rows_g.loc[valid_rows_g["payment_nature"] == "criminal", "amount_usd_float"].sum()

    paper_rows.append({
        "respondent_role": group_label,
        "n_incidents": n_incidents_g, "n_outcomes": n_outcomes_g, "n_with_amount": n_with_amount_g,
        "compensatory_usd": comp_usd, "penalty_usd": pen_usd, "criminal_usd": crim_usd,
    })

paper_table = pd.DataFrame(paper_rows)
print(f"\n8. Paper table (respondent_role x outcome/amount summary) -- as of {OUTCOMES_AS_OF}:")
print(paper_table.to_string(index=False))

print(f"\n9. OUTCOMES_AS_OF = {OUTCOMES_AS_OF}")

# Save everything. Sheet names kept short (Excel's 31-char limit).
xlsx_path = os.path.join(BATCH_DIR, "module3_respondent_payment_tables.xlsx")
with pd.ExcelWriter(xlsx_path) as writer:
    pd.DataFrame([{
        "n_outcome_rows": n_outcome_rows_c, "n_incidents": n_incidents_outcome,
        "n_sampled": n_sampled_c, "pct_of_sampled": pct_of_sampled, "as_of": OUTCOMES_AS_OF,
    }]).to_excel(writer, sheet_name="coverage", index=False)
    role_counts.rename_axis("respondent_role").reset_index(name="n").to_excel(
        writer, sheet_name="role_counts", index=False)
    ct_role_outcome.to_excel(writer, sheet_name="role_x_outcome")
    ct_role_lob.to_excel(writer, sheet_name="role_x_lob")
    ct_nature_outcome.to_excel(writer, sheet_name="nature_x_outcome")
    row_level.to_excel(writer, sheet_name="amt_row_level", index=False)
    incident_level.rename("total_amount").reset_index().to_excel(writer, sheet_name="amt_incident_level", index=False)
    respondent_level.to_excel(writer, sheet_name="amt_respondent_level", index=False)
    above_limit.to_excel(writer, sheet_name="above_cover_limit", index=False)
    multi.to_excel(writer, sheet_name="multi_outcome_incidents", index=False)
    paper_table.to_excel(writer, sheet_name="paper_table", index=False)
print(f"\nSaved {xlsx_path}")

paper_table_path = os.path.join(BATCH_DIR, "module3_table_respondent_payment.csv")
paper_table.to_csv(paper_table_path, index=False)
print(f"Saved {paper_table_path}")

# 6. Module 4 --- Insurability Assessment: Assessable Criteria (H3a-H3d)

Builds H3a-H3d from scratch: none of them existed in this notebook before this
section. Every paper-stated number below is treated as a target to reproduce
and cross-check, not as ground truth to copy: where this notebook's
computation disagrees, that disagreement is reported plainly rather than
reconciled.

H4 is tested in Module 5, which reuses the H3a-H3d constructs built here.

Hawkes/self-exciting process fitting is explicitly out of scope: AIID incident
dates are day-level only, ~30% of days in the primary window have more than
one incident, and a continuous-time triggering model on data with this many
tied event times risks mistaking same-day reporting clustering for genuine
causal triggering. (A discrete-time alternative is noted as future work in
the paper; not built here.)

In [ ]:
# H3a -- Intentionality (fortuity)
crosscheck_rows = []

intent_crosstab = classified["mit_intent"].fillna("(null/unclassified)").replace("", "(null/unclassified)").value_counts()
intent_crosstab_pct = (intent_crosstab / len(classified) * 100).round(1)
h3a_full = pd.DataFrame({"count": intent_crosstab, "pct_of_all": intent_crosstab_pct})
print(f"H3a: mit_intent crosstab, primary window (n={len(classified)})")
print(h3a_full.to_string())
print()

# Classified-only split (excludes null/unclassified and "Other").
n_unintentional = int((classified["mit_intent"] == "Unintentional").sum())
n_intentional = int((classified["mit_intent"] == "Intentional").sum())
n_intent_classified = n_unintentional + n_intentional
pct_unintentional = n_unintentional / n_intent_classified * 100 if n_intent_classified else float("nan")
pct_intentional = n_intentional / n_intent_classified * 100 if n_intent_classified else float("nan")

print(f"Classified-only (excluding null/Other): n = {n_intent_classified}")
print(f"  Unintentional: {n_unintentional} ({pct_unintentional:.1f}%)")
print(f"  Intentional:   {n_intentional} ({pct_intentional:.1f}%)")
print()

PAPER_H3A_UNINTENTIONAL = 564
PAPER_H3A_INTENTIONAL = 624
h3a_counts_match = (n_unintentional == PAPER_H3A_UNINTENTIONAL) and (n_intentional == PAPER_H3A_INTENTIONAL)
print("H3a cross-check vs. paper (unintentional=564, intentional=624, ~47.0%/52.0%):")
print(f"  notebook: unintentional={n_unintentional}, intentional={n_intentional} "
      f"({pct_unintentional:.1f}%/{pct_intentional:.1f}%)")
if h3a_counts_match:
    print("  MATCH")
else:
    print(f"  MISMATCH: paper states 564/624, notebook computes {n_unintentional}/{n_intentional}")
# Note: 564+624=1188, but 564/1188=47.5% not the paper's stated 47.0% -- the
# paper's percentages may use a different denominator (e.g. all classified
# incidents including null/Other, not just the intent-classified subset).
# Flagging this denominator ambiguity rather than silently picking one.
print("  NOTE: the paper's 47.0%/52.0% doesn't reduce to 564/1188 exactly (=47.5%/52.5%) --")
print("  possible denominator mismatch (all classified vs. intent-classified-only); unresolved.")
print()

crosscheck_rows.append({
    "hypothesis": "H3a", "metric": "Unintentional / Intentional counts",
    "paper_value": "564 / 624", "notebook_value": f"{n_unintentional} / {n_intentional}",
    "status": "MATCH" if h3a_counts_match else "MISMATCH",
})

h3a_out_path = os.path.join(BATCH_DIR, "module4_h3a_intent.csv")
h3a_full.to_csv(h3a_out_path)
print(f"Saved {h3a_out_path}")

In [ ]:
# H3b -- Insurable interest (diffuse harm)
print("Sample `harmed` values (first 15 non-empty):")
for h in classified.loc[classified["harmed"] != "", "harmed"].head(15):
    print(" ", repr(h))
print()
print("`harmed` uses the same '; '-delimited multi-value convention as")
print("`developer`/`deployer` (see names_join() in the batch-combine cell);")
print("splitting on ';' below.")
print()

classified["harmed_entities"] = classified["harmed"].apply(split_entities) if "split_entities" in dir() else classified["harmed"].apply(lambda t: [e.strip() for e in str(t).split(";") if e.strip()])

from collections import Counter

harmed_counts = Counter(e for entities in classified["harmed_entities"] for e in entities)
harmed_freq = pd.Series(harmed_counts, name="incident_count").sort_values(ascending=False)
print(f"H3b: ranked harmed-party frequency table (n={len(harmed_freq)} distinct categories):")
print(harmed_freq.to_string())
print()

DIFFUSE_PHRASES = ["epistemic integrity", "democratic integrity", "general public"]


def has_diffuse(entities):
    return any(any(p in e.lower() for p in DIFFUSE_PHRASES) for e in entities)


classified["has_diffuse_harm"] = classified["harmed_entities"].apply(has_diffuse)
has_harmed_field = classified["harmed"] != ""
n_with_harmed = int(has_harmed_field.sum())
n_diffuse = int((classified["has_diffuse_harm"] & has_harmed_field).sum())
pct_diffuse = n_diffuse / n_with_harmed * 100 if n_with_harmed else float("nan")

print(f"Incidents with a non-null harmed-party field: {n_with_harmed}")
print(f"Incidents with >=1 diffuse category (Epistemic Integrity / Democratic Integrity / "
      f"General Public): {n_diffuse} ({pct_diffuse:.1f}%)")
print()

print("Best (lowest-rank) match for each of the three named diffuse categories within the "
      "overall frequency table (a phrase like 'general public' can match many country-qualified "
      "variants, e.g. 'General public of Australia'; only the best match per phrase is kept):")
diffuse_ranks = {}
for phrase in DIFFUSE_PHRASES:
    matches = [name for name in harmed_freq.index if phrase in name.lower()]
    if matches:
        ranks = {name: int(harmed_freq.index.get_loc(name)) + 1 for name in matches}
        best_name = min(ranks, key=ranks.get)
        best_rank = ranks[best_name]
        diffuse_ranks[phrase] = best_rank
        print(f"  '{phrase}': best match = '{best_name}' (count={int(harmed_freq[best_name])}, "
              f"rank={best_rank} of {len(harmed_freq)}); {len(matches)} total variant(s) matching this phrase")
    else:
        print(f"  No entity matching '{phrase}' found in this dataset's harmed-party field.")
        diffuse_ranks[phrase] = None

NEAR_TOP_THRESHOLD = 20
all_near_top = all(r is not None and r <= NEAR_TOP_THRESHOLD for r in diffuse_ranks.values())
print()
print("Paper's qualitative claim: Epistemic Integrity, Democratic Integrity, and General Public "
      "are the most frequently recorded diffuse categories (near the top of the frequency table).")
if all_near_top:
    print("  MATCH")
else:
    print(f"  MISMATCH: not all three rank within the top {NEAR_TOP_THRESHOLD} -- {diffuse_ranks}")
print()
print(f"  NEW (not yet stated in paper text): combined diffuse-category share = "
      f"{pct_diffuse:.1f}% of incidents with a harmed-party field")

crosscheck_rows.append({
    "hypothesis": "H3b", "metric": "Top diffuse harmed-party categories",
    "paper_value": "Epistemic Integrity, Democratic Integrity, General Public (qualitative)",
    "notebook_value": str(diffuse_ranks), "status": "MATCH" if all_near_top else "MISMATCH",
})
crosscheck_rows.append({
    "hypothesis": "H3b", "metric": "% incidents with diffuse harmed-party category",
    "paper_value": "not stated", "notebook_value": f"{pct_diffuse:.1f}%", "status": "NEW",
})

h3b_out_path = os.path.join(BATCH_DIR, "module4_h3b_harmed_party.csv")
harmed_freq.rename_axis("harmed_party_category").reset_index().to_csv(h3b_out_path, index=False)
print(f"\nSaved {h3b_out_path}")

In [ ]:
# Before H3c/H3d: check for existing entity-parsing logic.
#
# Table 1's Concentration (HHI) variable description distinguishes (a) named
# corporate entities and (b) all entity labels including generic category
# labels (e.g. "Deepfake Technology Developers") -- this would imply a more
# careful entity-cleaning approach. Searched this notebook (every cell above,
# including Module 1's Attribution Gap / Entity Recurrence work): NO such
# pipeline exists anywhere in this notebook. Building a first-pass parser
# instead: split the semicolon-joined `developer`/`deployer` fields (this
# project's established multi-entity delimiter -- see names_join() in the
# batch-combine cell) and strip whitespace. A category-label vs. named-entity
# distinction is NOT implemented here -- note this before citing these
# figures alongside Table 1's HHI variable.
from collections import Counter


def split_entities(text):
    return [e.strip() for e in str(text).split(";") if e.strip()]


classified["developer_entities"] = classified["developer"].apply(split_entities)
classified["deployer_entities"] = classified["deployer"].apply(split_entities)

deployer_entity_counts = Counter(e for entities in classified["deployer_entities"] for e in entities)
entity_counts = pd.Series(deployer_entity_counts, name="incident_count").sort_values(ascending=False)

print(f"Parsed developer/deployer entities from {len(classified)} primary-window classified incidents.")
print(f"  {len(entity_counts)} unique deployer entities, {int(entity_counts.sum())} total entity-incident mentions")
print("Reused for H3c (attribution gap) and H3d (concentration) below.")

In [ ]:
# Entity codebook (named vs. generic vs. unknown) -- STEP 1 ONLY.
#
# H3c/H3d and the H4 comparisons currently pool every distinct
# developer_entities/deployer_entities label as if it were a single entity.
# Many labels are generic descriptions of a TYPE of actor, not a specific
# one (e.g. "Scammers" (182 mentions), "Deepfake technology developers"
# (337), "Synthetic audio generation technology developers" (191),
# "Unknown" (86)) -- pooling these as single entities inflates deployer
# concentration (H3d) and developer-deployer separation (H3c),
# disproportionately in the gaining-share group where this kind of generic
# attribution is more common. The cell above already flagged that this
# named-vs-generic distinction was never built.
#
# This cell builds ONLY the codebook (label, role, n_mentions, a pre-filled
# entity_class, and a pre-filled canonical_name) and saves it for human
# review. Recomputing H3c/H3d/H4 under alternative entity-class specs is a
# SEPARATE, LATER step -- do NOT proceed to it until the saved codebook has
# been reviewed by hand. entity_class and canonical_name below are a
# first-pass keyword rule, not a verified coding.
import re

developer_label_counts = Counter(e for entities in classified["developer_entities"] for e in entities)
deployer_label_counts = Counter(e for entities in classified["deployer_entities"] for e in entities)
all_entity_labels = sorted(set(developer_label_counts) | set(deployer_label_counts))

codebook_rows = []
for label in all_entity_labels:
    dev_n = developer_label_counts.get(label, 0)
    dep_n = deployer_label_counts.get(label, 0)
    if dev_n > 0 and dep_n > 0:
        role = "both"
    elif dev_n > 0:
        role = "developer"
    else:
        role = "deployer"
    codebook_rows.append({
        "label": label, "role": role,
        "n_mentions_developer": dev_n, "n_mentions_deployer": dep_n,
        "n_mentions": dev_n + dep_n,
    })

entity_codebook = pd.DataFrame(codebook_rows)

# --- Pre-fill entity_class via a keyword rule (first pass -- needs human
# review; deliberately conservative word-boundary matches on generic
# actor-type nouns, e.g. "...developers", "...actors", "Scammers", rather
# than substring matches that could false-positive on a named entity whose
# proper name happens to contain one of these words). ---
UNKNOWN_KEYWORD = re.compile(r"unknown", re.IGNORECASE)
GENERIC_KEYWORDS = re.compile(
    r"\b(developers?|actors?|creators?|scammers?|students?|cybercriminals?|"
    r"criminals?|users?|companies|organi[sz]ations?|groups?|operators?|"
    r"individuals?|researchers?|employees?|workers?|officials?|agencies|"
    r"entities|parties|providers?|firms?|teachers?|customers?|patients?|"
    r"drivers?|attackers?|hackers?)\b",
    re.IGNORECASE,
)


def prefill_entity_class(label):
    if UNKNOWN_KEYWORD.search(label):
        return "unknown"
    if GENERIC_KEYWORDS.search(label):
        return "generic"
    return "named"


entity_codebook["entity_class"] = entity_codebook["label"].apply(prefill_entity_class)

# --- Pre-fill canonical_name: explicit alias merges, else the label itself.
# Every merge in this dict is listed (and whether it actually matched
# anything in this dataset) in the printout below. ---
CANONICAL_MERGES = {
    "Facebook": "Meta",
    "X (Twitter)": "X",
    "Twitter": "X",
    "Perplexity AI": "Perplexity",
    "ChatGPT": "OpenAI",
}
entity_codebook["canonical_name"] = (
    entity_codebook["label"].map(CANONICAL_MERGES).fillna(entity_codebook["label"])
)

present_labels = set(entity_codebook["label"])
applied_merges = {k: v for k, v in CANONICAL_MERGES.items() if k in present_labels}
not_found_merges = {k: v for k, v in CANONICAL_MERGES.items() if k not in present_labels}
print("Canonical-name merges applied (label -> canonical_name, only those present in this dataset):")
if applied_merges:
    for k, v in applied_merges.items():
        print(f"  {k!r} -> {v!r}")
else:
    print("  (none of the predefined merge labels were found in this dataset)")
if not_found_merges:
    print(f"Predefined merges NOT applied (label not present in this dataset): {not_found_merges}")

# --- Flag every row for human review; sort so >=2-mention labels are at
# the top (singletons, the bulk of the table, sort to the bottom). ---
entity_codebook["needs_review"] = True
entity_codebook = entity_codebook.sort_values("n_mentions", ascending=False).reset_index(drop=True)

print()
print(f"Entity codebook: {len(entity_codebook)} distinct labels "
      f"({int((entity_codebook['n_mentions'] >= 2).sum())} with >=2 mentions, "
      f"{int((entity_codebook['n_mentions'] == 1).sum())} singletons).")
print()
print("Pre-filled entity_class breakdown (BEFORE human review):")
print(entity_codebook["entity_class"].value_counts().to_string())
print()
print("Top 20 labels by n_mentions:")
print(entity_codebook.head(20).to_string(index=False))

entity_codebook_path = os.path.join(BATCH_DIR, "module4_entity_codebook.csv")
entity_codebook.to_csv(entity_codebook_path, index=False)
print(f"\nSaved {entity_codebook_path}")
print()
print("STOP: do not proceed to recomputing H3c/H3d/H4 under alternative entity")
print("specs until this codebook has been reviewed by hand (entity_class and")
print("canonical_name above are a first-pass keyword rule, not a verified coding).")

In [ ]:
# H3c -- Information asymmetry / allocation ambiguity (attribution gap)
has_both = classified.apply(
    lambda r: len(r["developer_entities"]) > 0 and len(r["deployer_entities"]) > 0, axis=1
)
both_df = classified[has_both].copy()
n_both = len(both_df)


def is_disjoint(row):
    dev_set = {d.lower() for d in row["developer_entities"]}
    dep_set = {d.lower() for d in row["deployer_entities"]}
    return dev_set.isdisjoint(dep_set)


both_df["disjoint"] = both_df.apply(is_disjoint, axis=1)
n_disjoint = int(both_df["disjoint"].sum())
n_overlap = n_both - n_disjoint
pct_disjoint = n_disjoint / n_both * 100 if n_both else float("nan")
pct_overlap = n_overlap / n_both * 100 if n_both else float("nan")

print("H3c: attribution gap, primary window")
print(f"  n incidents with both developer and deployer populated: {n_both}")
print(f"  Disjoint (no shared entity):     {n_disjoint} ({pct_disjoint:.1f}%)")
print(f"  Overlapping (>=1 shared entity): {n_overlap} ({pct_overlap:.1f}%)")
print(f"  (disjoint% + overlap% = {pct_disjoint + pct_overlap:.1f}%, should be 100.0)")
print()

PAPER_H3C_DISJOINT_PCT = 58.0
h3c_match = abs(pct_disjoint - PAPER_H3C_DISJOINT_PCT) < 0.05
print(f"H3c cross-check vs. paper ({PAPER_H3C_DISJOINT_PCT}% disjoint):")
print(f"  notebook computes {pct_disjoint:.1f}% disjoint")
if h3c_match:
    print("  MATCH")
else:
    print(f"  MISMATCH: paper states {PAPER_H3C_DISJOINT_PCT}%, notebook computes {pct_disjoint:.1f}%")
print()
print(f"  For reference: overlap ({pct_overlap:.1f}%) vs. the ~45% same-party figure Richards, "
      f"Benn & Zilka (2025) report on a different, earlier AIID snapshot -- not expected to match "
      f"exactly (different snapshot/window), reported for context only.")

crosscheck_rows.append({
    "hypothesis": "H3c", "metric": "% developer-deployer disjoint (attribution gap)",
    "paper_value": f"{PAPER_H3C_DISJOINT_PCT}%", "notebook_value": f"{pct_disjoint:.1f}%",
    "status": "MATCH" if h3c_match else "MISMATCH",
})

h3c_summary = pd.DataFrame([{
    "n_both_populated": n_both, "n_disjoint": n_disjoint, "n_overlapping": n_overlap,
    "pct_disjoint": round(pct_disjoint, 1), "pct_overlapping": round(pct_overlap, 1),
}])
h3c_out_path = os.path.join(BATCH_DIR, "module4_h3c_attribution_gap.csv")
h3c_summary.to_csv(h3c_out_path, index=False)
print(f"\nSaved {h3c_out_path}")

In [ ]:
# H3d -- Loss independence (entity concentration): descriptive summary
n_entities_total = len(entity_counts)
n_singleton_total = int((entity_counts == 1).sum())
pct_singleton_total = n_singleton_total / n_entities_total * 100

n_ge5_total = int((entity_counts >= 5).sum())
total_mentions = int(entity_counts.sum())

# Two distinct denominators -- kept as separate, clearly labelled metrics so
# they can't be confused with each other:
#   mentions_from_ge5 / total_mentions: share of entity-INCIDENT MENTIONS
#     contributed by entities appearing >=5 times (entity-level; one
#     incident with 3 deployer entities contributes up to 3 mentions).
#   incidents_from_ge5 / len(classified): share of INCIDENTS involving at
#     least one entity appearing >=5 times (incident-level). This is what
#     the paper's figure describes -- the previous version of this cell
#     used the mentions denominator for this (a bug: 35.8%, a false
#     MISMATCH against the paper's 53.9%).
mentions_from_ge5 = int(entity_counts[entity_counts >= 5].sum())
pct_mentions_from_ge5 = mentions_from_ge5 / total_mentions * 100

heavy = set(entity_counts[entity_counts >= 5].index)
incidents_from_ge5 = int(classified["deployer_entities"].apply(lambda l: any(e in heavy for e in l)).sum())
pct_incidents_from_ge5 = incidents_from_ge5 / len(classified) * 100

print("H3d: entity concentration, primary window")
print(f"  {n_entities_total} unique deployer entities, {total_mentions} total entity-incident mentions")
print(f"  Entities appearing exactly once: {n_singleton_total} ({pct_singleton_total:.1f}%)")
print(f"  Entities appearing >=5 times: {n_ge5_total}")
print(f"    -> {mentions_from_ge5} entity-incident MENTIONS from these entities "
      f"({pct_mentions_from_ge5:.1f}% of all mentions)")
print(f"    -> {incidents_from_ge5} INCIDENTS involve at least one such entity "
      f"({pct_incidents_from_ge5:.1f}% of all {len(classified)} classified incidents)")
print()

h3d_concentration_summary = pd.DataFrame([{
    "n_unique_entities": n_entities_total, "total_mentions": total_mentions,
    "n_singleton_entities": n_singleton_total, "pct_singleton_entities": round(pct_singleton_total, 1),
    "n_entities_ge5": n_ge5_total,
    "mentions_from_ge5": mentions_from_ge5, "pct_mentions_from_ge5": round(pct_mentions_from_ge5, 1),
    "incidents_from_ge5": incidents_from_ge5, "pct_incidents_from_ge5": round(pct_incidents_from_ge5, 1),
}])
print(h3d_concentration_summary.to_string(index=False))
print()

PAPER_H3D_SINGLETON_PCT = 89.5   # was 88.3
PAPER_H3D_N_GE5 = 45             # was 59
PAPER_H3D_PCT_GE5 = 53.9         # was 54.5 -- compared against the INCIDENT-level
                                  # share below, not the mention-level share (the fixed bug)

singleton_match = abs(pct_singleton_total - PAPER_H3D_SINGLETON_PCT) < 0.1
n_ge5_match = n_ge5_total == PAPER_H3D_N_GE5
pct_ge5_match = abs(pct_incidents_from_ge5 - PAPER_H3D_PCT_GE5) < 0.1

print(f"H3d cross-check vs. paper ({PAPER_H3D_SINGLETON_PCT}% singleton; {PAPER_H3D_N_GE5} entities "
      f">=5x accounting for {PAPER_H3D_PCT_GE5}% of INCIDENTS):")
singleton_verdict = "MATCH" if singleton_match else "MISMATCH"
n_ge5_verdict = "MATCH" if n_ge5_match else "MISMATCH"
pct_ge5_verdict = "MATCH" if pct_ge5_match else "MISMATCH"
print(f"  Singleton %: notebook={pct_singleton_total:.1f}%, paper={PAPER_H3D_SINGLETON_PCT}% -> {singleton_verdict}")
print(f"  N entities >=5x: notebook={n_ge5_total}, paper={PAPER_H3D_N_GE5} -> {n_ge5_verdict}")
print(f"  % of INCIDENTS involving an entity >=5x: notebook={pct_incidents_from_ge5:.1f}%, "
      f"paper={PAPER_H3D_PCT_GE5}% -> {pct_ge5_verdict}")
print(f"  (for reference, % of entity-MENTIONS from these entities: {pct_mentions_from_ge5:.1f}% -- "
      f"a different, entity-level denominator, not compared against the paper figure)")

crosscheck_rows.append({
    "hypothesis": "H3d", "metric": "% entities appearing exactly once",
    "paper_value": f"{PAPER_H3D_SINGLETON_PCT}%", "notebook_value": f"{pct_singleton_total:.1f}%",
    "status": "MATCH" if singleton_match else "MISMATCH",
})
crosscheck_rows.append({
    "hypothesis": "H3d", "metric": "N entities appearing >=5 times",
    "paper_value": str(PAPER_H3D_N_GE5), "notebook_value": str(n_ge5_total),
    "status": "MATCH" if n_ge5_match else "MISMATCH",
})
crosscheck_rows.append({
    "hypothesis": "H3d", "metric": "% of incidents involving an entity appearing >=5 times",
    "paper_value": f"{PAPER_H3D_PCT_GE5}%", "notebook_value": f"{pct_incidents_from_ge5:.1f}%",
    "status": "MATCH" if pct_ge5_match else "MISMATCH",
})

h3d_out_path = os.path.join(BATCH_DIR, "module4_h3d_concentration.csv")
h3d_concentration_summary.to_csv(h3d_out_path, index=False)
print(f"\nSaved {h3d_out_path}")

In [ ]:
# Dispersion test: Poisson GLM Pearson chi-square statistic, cross-checked
# against a Negative Binomial fit's estimated dispersion parameter (alpha).
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2 as chi2_dist
from statsmodels.discrete.discrete_model import NegativeBinomial as NB_MLE


def overdispersion_test(counts, label):
    """Poisson dispersion test + Negative Binomial alpha cross-check.

    variance/mean > 1 with a significant Pearson dispersion statistic and
    an NB alpha whose 95% CI excludes 0 both indicate overdispersion; they
    should agree on the qualitative conclusion.
    """
    counts = np.asarray(counts, dtype=float)
    n = len(counts)
    mean_count = counts.mean()
    var_count = counts.var(ddof=1)
    ratio = var_count / mean_count

    counts_df = pd.DataFrame({"count": counts})
    poisson_model = smf.glm("count ~ 1", data=counts_df, family=sm.families.Poisson()).fit()
    pearson_chi2 = float(np.sum(poisson_model.resid_pearson ** 2))
    dispersion_stat = pearson_chi2 / poisson_model.df_resid
    p_value = chi2_dist.sf(pearson_chi2, poisson_model.df_resid)

    # NegativeBinomial GLM via the formula API only fits with a FIXED
    # alpha (default 1.0) -- it does not estimate it. The MLE-estimated
    # alpha (with a real confidence interval) requires the discrete model.
    exog = np.ones((n, 1))
    nb_model = NB_MLE(counts, exog).fit(disp=0)
    nb_alpha = float(nb_model.params[-1])
    nb_ci = nb_model.conf_int()
    nb_alpha_lo, nb_alpha_hi = float(nb_ci[-1][0]), float(nb_ci[-1][1])

    print(f"--- {label} (n = {n} entities) ---")
    print(f"  Mean count:             {mean_count:.4f}")
    print(f"  Variance:               {var_count:.4f}")
    print(f"  Variance-to-mean ratio: {ratio:.4f}")
    print(f"  Dispersion stat (Pearson chi2/df): {dispersion_stat:.4f}")
    print(f"  p-value (chi2 test for equidispersion): {p_value:.3g}")
    print(f"  Negative Binomial alpha: {nb_alpha:.4f} (95% CI [{nb_alpha_lo:.4f}, {nb_alpha_hi:.4f}])")
    print(f"  Poisson AIC = {poisson_model.aic:.1f}  |  Negative Binomial AIC = {nb_model.aic:.1f}")

    poisson_overdispersed = ratio > 1 and p_value < 0.05
    nb_overdispersed = nb_alpha_lo > 0
    agree = poisson_overdispersed == nb_overdispersed
    overdispersed = poisson_overdispersed and nb_overdispersed

    poisson_verdict = "overdispersed" if poisson_overdispersed else "not overdispersed"
    agree_verdict = "agree" if agree else "DISAGREE -- inspect before reporting"
    overall_verdict = "OVERDISPERSED" if overdispersed else "NOT clearly overdispersed"
    support_verdict = "supports" if overdispersed else "does not support"
    print(f"  Poisson dispersion test: {poisson_verdict}"
          f" | NB alpha CI excludes 0: {nb_overdispersed}"
          f" | tests {agree_verdict}")
    print(f"  -> {overall_verdict}: this "
          f"{support_verdict} H3d's aggregation/concentration-potential claim.")
    print()

    return {
        "variant": label, "n_entities": n, "mean": mean_count, "variance": var_count,
        "ratio": ratio, "dispersion_stat": dispersion_stat, "p_value": p_value,
        "nb_alpha": nb_alpha, "nb_alpha_ci_low": nb_alpha_lo, "nb_alpha_ci_high": nb_alpha_hi,
    }


overdispersion_results = []
overdispersion_results.append(
    overdispersion_test(entity_counts.to_numpy(), "Full population (all deployer entities)")
)

# Repeat entities only (>=2 incidents), excluding the singleton-entity mass
# that can mechanically produce overdispersion on its own.
repeat_counts = entity_counts[entity_counts >= 2]
overdispersion_results.append(
    overdispersion_test(repeat_counts.to_numpy(), "Repeat entities only (>=2 incidents)")
)

overdispersion_df = pd.DataFrame(overdispersion_results)
print(overdispersion_df.to_string(index=False))

In [ ]:
overdispersion_path = os.path.join(BATCH_DIR, "module4_h3d_overdispersion.csv")
overdispersion_df.to_csv(overdispersion_path, index=False)
print(f"Saved {overdispersion_path}")

In [ ]:
# Final consolidated cross-check summary across H3a-H3d.
# Overdispersion results are folded in here too (appended to crosscheck_rows
# so the whole H3a-H3d picture is in one table).
crosscheck_rows.append({
    "hypothesis": "H3d", "metric": "Overdispersion (full population)",
    "paper_value": "overdispersed (qualitative)",
    "notebook_value": f"ratio={overdispersion_results[0]['ratio']:.2f}, p={overdispersion_results[0]['p_value']:.3g}",
    "status": "MATCH" if (overdispersion_results[0]["ratio"] > 1
                           and overdispersion_results[0]["p_value"] < 0.05
                           and overdispersion_results[0]["nb_alpha_ci_low"] > 0) else "MISMATCH",
})
crosscheck_rows.append({
    "hypothesis": "H3d", "metric": "Overdispersion (repeat entities only)",
    "paper_value": "overdispersed (qualitative)",
    "notebook_value": f"ratio={overdispersion_results[1]['ratio']:.2f}, p={overdispersion_results[1]['p_value']:.3g}",
    "status": "MATCH" if (overdispersion_results[1]["ratio"] > 1
                           and overdispersion_results[1]["p_value"] < 0.05
                           and overdispersion_results[1]["nb_alpha_ci_low"] > 0) else "MISMATCH",
})

crosscheck_summary = pd.DataFrame(crosscheck_rows)
print("Final consolidated cross-check table (H3a-H3d):")
print(crosscheck_summary.to_string(index=False))

crosscheck_path = os.path.join(BATCH_DIR, "module4_crosscheck_summary.csv")
crosscheck_summary.to_csv(crosscheck_path, index=False)

print()
print("All saved paths:")
for p in [h3a_out_path, h3b_out_path, h3c_out_path, h3d_out_path, overdispersion_path, crosscheck_path]:
    print(" ", p)

# 7. Module 5 --- Risk-Direction Test (H4)

H4: risk categories gaining share score worse against the assessable
Berliner criteria (H3a-H3d) than the categories whose relative share is
declining. The main specification (Spec B) defines the gaining/losing
groups by each domain's change in share of `classified` incidents
between the pre- and post-`BREAK_YEAR` periods. Two robustness
specifications follow: Spec A groups by Module 1's Mann-Kendall count
trend instead of share change, and Spec C restricts to the domains with
the largest share movement (>=5 percentage points), excluding small
movers. Reuses `classified`, `mit_risk_domain`, `mit_intent` (H3a),
`has_diffuse_harm` (H3b), `developer_entities`/`deployer_entities`
(H3c/H3d), and `overdispersion_test()` (H3d), all built in Module 1/4
above -- this section must run after Module 4.

In [ ]:
# ── MODULE 5 : Cell 5.1 -- domain share change (pre/post BREAK_YEAR) and H4 group specs ──
#
# H4, as stated, is about SHARE: "risk categories gaining share score worse
# against the assessable Berliner criteria than the categories whose
# relative share is declining." Module 1's Mann-Kendall test (cell 18) is a
# COUNT trend test, not a share test -- AI Safety (41.3% -> 13.1%) and
# Discrimination (31.3% -> 10.1%) are flat in count but losing share (that
# pattern is H1a's finding, not H4's), and Privacy & Security's count rises
# while its share falls (7.1% -> 6.3%). Groups are therefore defined by
# share change, computed below, not by the count-trend result the earlier
# version of this module used.
#
# Objects reused (all built in Module 1/4 above): `classified`,
# `mit_risk_domain`, `BREAK_YEAR`, `domains`.

pre = classified[classified["year"] < BREAK_YEAR]
post = classified[classified["year"] >= BREAK_YEAR]

pre_pct = (pre["mit_risk_domain"].value_counts() / len(pre) * 100).reindex(domains).fillna(0.0)
post_pct = (post["mit_risk_domain"].value_counts() / len(post) * 100).reindex(domains).fillna(0.0)
share_change = post_pct - pre_pct

share_table = pd.DataFrame({
    "pre_pct": pre_pct.round(1), "post_pct": post_pct.round(1), "change_pp": share_change.round(1),
}).reindex(domains)
print(f"Domain share of `classified`, pre-BREAK_YEAR (n={len(pre)}, year < {BREAK_YEAR}) "
      f"vs. post-BREAK_YEAR (n={len(post)}, year >= {BREAK_YEAR}):")
print(share_table.to_string())
print()

# --- Spec B (main): group by sign of the share change computed above. ---
spec_b = {}
for d in domains:
    delta = share_change[d]
    if delta > 0:
        spec_b[d] = "gaining"
    elif delta < 0:
        spec_b[d] = "losing"
    else:
        spec_b[d] = "excluded"

# --- Spec A (robustness, count trend): Module 1 cell 18's Mann-Kendall
# finding is a count-level result, not re-derivable from `classified` alone
# the way share change is -- so the three count-increasing domains are
# named directly here, matching cell 18's cross-checked finding. All 7
# domains are covered (no exclusions), since Module 1 only distinguishes
# increasing vs. not-increasing. ---
GAINING_COUNT_TREND = [
    "4. Malicious Actors & Misuse",
    "3. Misinformation",
    "2. Privacy & Security",
]
_missing_a = [d for d in GAINING_COUNT_TREND if d not in domains]
if _missing_a:
    print(f"WARNING: Spec A's GAINING_COUNT_TREND domains not found in `domains`: {_missing_a}")
spec_a = {d: ("gaining" if d in GAINING_COUNT_TREND else "losing") for d in domains}

# --- Spec C (robustness, core movers only): same share_change as Spec B,
# but only domains moving >=5pp in either direction are grouped; smaller
# movers (e.g. Privacy & Security's -0.8pp) are excluded rather than forced
# into a bucket. ---
CORE_MOVER_THRESHOLD_PP = 5.0
spec_c = {}
for d in domains:
    delta = share_change[d]
    if delta >= CORE_MOVER_THRESHOLD_PP:
        spec_c[d] = "gaining"
    elif delta <= -CORE_MOVER_THRESHOLD_PP:
        spec_c[d] = "losing"
    else:
        spec_c[d] = "excluded"

H4_SPECS = {
    "B_share_change": spec_b,
    "A_count_trend": spec_a,
    "C_core_movers": spec_c,
}

print("H4 group assignment per domain, by spec:")
spec_table = pd.DataFrame(H4_SPECS).reindex(domains)
print(spec_table.to_string())
print()

for spec_name, spec_map in H4_SPECS.items():
    mapped = classified["mit_risk_domain"].map(spec_map)
    counts = mapped.value_counts()
    n_g = int(counts.get("gaining", 0))
    n_l = int(counts.get("losing", 0))
    n_x = int(counts.get("excluded", 0))
    extra = f", n excluded={n_x}" if n_x else ""
    print(f"{spec_name}: n gaining={n_g}, n losing={n_l}{extra}")

In [ ]:
# ── MODULE 5 : Cell 5.2 -- h4_compare() helper + run across all three specs ──
#
# One helper, reused for every spec/criterion, instead of separate
# per-criterion cells. overdispersion_test(counts, label) is defined in
# Module 4 (cell 44) -- it must already have run in this kernel session.
from collections import Counter

import numpy as np
from scipy.stats import chi2_contingency

# Shared across every H3d bootstrap call below (all three specs draw from
# this same rng stream) so the whole cell is reproducible under one seed.
B_BOOT = 2000
rng = np.random.default_rng(42)


def _h3c_disjoint_subset(df):
    """Re-uses the case-insensitive developer/deployer isdisjoint logic from
    the Module 4 H3c cell, restricted to incidents with both fields populated."""
    has_both = df.apply(
        lambda r: len(r["developer_entities"]) > 0 and len(r["deployer_entities"]) > 0, axis=1
    )
    sub = df[has_both].copy()

    def is_disjoint(row):
        dev_set = {d.lower() for d in row["developer_entities"]}
        dep_set = {d.lower() for d in row["deployer_entities"]}
        return dev_set.isdisjoint(dep_set)

    sub["disjoint"] = sub.apply(is_disjoint, axis=1)
    return sub


def _entity_counts(df):
    counts = Counter(e for entities in df["deployer_entities"] for e in entities)
    return np.array(list(counts.values()), dtype=float)


def _dispersion_ratio(df):
    """variance/mean (ddof=1) of per-entity deployer counts within df."""
    counts = _entity_counts(df)
    if len(counts) < 2:
        return float("nan")
    return counts.var(ddof=1) / counts.mean()


def _bootstrap_ratio_diff(df_gain, df_lose, b=B_BOOT):
    """Bootstrap the difference in dispersion ratios (gaining - losing).

    Resamples INCIDENTS (rows), with replacement, at the same n as each
    group -- not entities directly -- since the incident is the sampling
    unit and one incident can list several deployers. Per-entity counts are
    rebuilt from each replicate's `deployer_entities` before computing the
    ratio.
    """
    gain_idx = df_gain.index.to_numpy()
    lose_idx = df_lose.index.to_numpy()
    diffs = np.empty(b)
    for i in range(b):
        boot_gain = df_gain.loc[rng.choice(gain_idx, size=len(gain_idx), replace=True)]
        boot_lose = df_lose.loc[rng.choice(lose_idx, size=len(lose_idx), replace=True)]
        diffs[i] = _dispersion_ratio(boot_gain) - _dispersion_ratio(boot_lose)
    return diffs


def h4_compare(df_gain, df_lose):
    """Compares a gaining-share and a losing-share incident subset across
    H3a-H3d. Returns {criterion: {gaining_value, losing_value, statistic,
    p_value, adverse}}; adverse=True means the gaining group scores worse
    (less insurable) on that criterion, matching H4's prediction."""
    results = {}

    # H3a -- % Intentional, restricted to classified Intentional/Unintentional rows.
    g_a = df_gain[df_gain["mit_intent"].isin(["Intentional", "Unintentional"])]
    l_a = df_lose[df_lose["mit_intent"].isin(["Intentional", "Unintentional"])]
    table_a = pd.DataFrame({
        "gaining": [int((g_a["mit_intent"] == "Intentional").sum()),
                    int((g_a["mit_intent"] == "Unintentional").sum())],
        "losing": [int((l_a["mit_intent"] == "Intentional").sum()),
                   int((l_a["mit_intent"] == "Unintentional").sum())],
    }, index=["Intentional", "Unintentional"])
    chi2_a, p_a, _, _ = chi2_contingency(table_a.T)
    pct_g_a = table_a.loc["Intentional", "gaining"] / table_a["gaining"].sum() * 100
    pct_l_a = table_a.loc["Intentional", "losing"] / table_a["losing"].sum() * 100
    results["H3a"] = {
        "gaining_value": pct_g_a, "losing_value": pct_l_a,
        "statistic": chi2_a, "p_value": p_a, "p_display": f"{p_a:.2e}",
        "adverse": pct_g_a > pct_l_a, "test": "chi2 2x2", "ci_low": None, "ci_high": None,
    }

    # H3b -- % has_diffuse_harm, restricted to rows with a non-null harmed field.
    g_b = df_gain[df_gain["harmed"] != ""]
    l_b = df_lose[df_lose["harmed"] != ""]
    table_b = pd.DataFrame({
        "gaining": [int(g_b["has_diffuse_harm"].sum()), int((~g_b["has_diffuse_harm"]).sum())],
        "losing": [int(l_b["has_diffuse_harm"].sum()), int((~l_b["has_diffuse_harm"]).sum())],
    }, index=["diffuse", "not_diffuse"])
    chi2_b, p_b, _, _ = chi2_contingency(table_b.T)
    pct_g_b = table_b.loc["diffuse", "gaining"] / table_b["gaining"].sum() * 100
    pct_l_b = table_b.loc["diffuse", "losing"] / table_b["losing"].sum() * 100
    results["H3b"] = {
        "gaining_value": pct_g_b, "losing_value": pct_l_b,
        "statistic": chi2_b, "p_value": p_b, "p_display": f"{p_b:.2e}",
        "adverse": pct_g_b > pct_l_b, "test": "chi2 2x2", "ci_low": None, "ci_high": None,
    }

    # H3c -- % developer-deployer disjoint (attribution gap).
    g_c = _h3c_disjoint_subset(df_gain)
    l_c = _h3c_disjoint_subset(df_lose)
    table_c = pd.DataFrame({
        "gaining": [int(g_c["disjoint"].sum()), int((~g_c["disjoint"]).sum())],
        "losing": [int(l_c["disjoint"].sum()), int((~l_c["disjoint"]).sum())],
    }, index=["disjoint", "overlap"])
    chi2_c, p_c, _, _ = chi2_contingency(table_c.T)
    pct_g_c = table_c.loc["disjoint", "gaining"] / table_c["gaining"].sum() * 100
    pct_l_c = table_c.loc["disjoint", "losing"] / table_c["losing"].sum() * 100
    results["H3c"] = {
        "gaining_value": pct_g_c, "losing_value": pct_l_c,
        "statistic": chi2_c, "p_value": p_c, "p_display": f"{p_c:.2e}",
        "adverse": pct_g_c > pct_l_c, "test": "chi2 2x2", "ci_low": None, "ci_high": None,
    }

    # H3d -- entity concentration / overdispersion. Each group's own
    # Poisson-equidispersion test (via overdispersion_test) only shows that
    # group IS overdispersed -- it says nothing about whether the gaining
    # group differs from the losing group. The actual between-group test is
    # a bootstrap of the difference in dispersion ratios, with incidents
    # (not entities) as the resampling unit.
    counts_g = _entity_counts(df_gain)
    counts_l = _entity_counts(df_lose)
    res_g = overdispersion_test(counts_g, "H4 gaining group")
    res_l = overdispersion_test(counts_l, "H4 losing group")
    cis_overlap = (res_g["nb_alpha_ci_low"] <= res_l["nb_alpha_ci_high"]
                   and res_l["nb_alpha_ci_low"] <= res_g["nb_alpha_ci_high"])
    print(f"  NB alpha: gaining={res_g['nb_alpha']:.4f} (95% CI [{res_g['nb_alpha_ci_low']:.4f}, "
          f"{res_g['nb_alpha_ci_high']:.4f}]), losing={res_l['nb_alpha']:.4f} (95% CI "
          f"[{res_l['nb_alpha_ci_low']:.4f}, {res_l['nb_alpha_ci_high']:.4f}])")
    print(f"  NB alpha CIs {'overlap' if cis_overlap else 'do NOT overlap'} -- the dispersion "
          f"ratio (not alpha) carries the gaining-vs-losing comparison here.")

    observed_diff = res_g["ratio"] - res_l["ratio"]
    boot_diffs = _bootstrap_ratio_diff(df_gain, df_lose)
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    n_le_zero = int((boot_diffs <= 0).sum())
    p_boot = n_le_zero / B_BOOT
    # One-sided bootstrap p: share of replicates with diff <= 0. Displayed as
    # "< 1/B_BOOT" rather than 0 when no replicate fell at or below 0 -- the
    # true p is bounded by, not equal to, zero at finite B -- but the
    # numeric p_value stored below is kept as a float (the 1/B_BOOT upper
    # bound in that case) so h4_summary["p_value"] stays a numeric column
    # instead of silently becoming object-dtype.
    p_boot_report = f"< {1 / B_BOOT:.4f}" if n_le_zero == 0 else p_boot
    p_value_numeric = p_boot if n_le_zero > 0 else 1 / B_BOOT

    print(f"  Bootstrap diff in dispersion ratio (gaining - losing), incident resampling, "
          f"B={B_BOOT}: observed={observed_diff:.4f}, 95% CI=[{ci_low:.4f}, {ci_high:.4f}], "
          f"one-sided p (share of replicates <= 0) = "
          f"{p_boot_report if isinstance(p_boot_report, str) else f'{p_boot_report:.4f}'}")

    results["H3d"] = {
        "gaining_value": res_g["ratio"], "losing_value": res_l["ratio"],
        "statistic": observed_diff, "p_value": p_value_numeric, "p_display": p_boot_report,
        "ci_low": ci_low, "ci_high": ci_high,
        "gain_equidisp_p": res_g["p_value"], "lose_equidisp_p": res_l["p_value"],
        "adverse": ci_low > 0,
        "test": "bootstrap diff in dispersion ratio (incident resampling, B=2000)",
    }

    return results


h4_results_by_spec = {}
for spec_name, spec_map in H4_SPECS.items():
    group = classified["mit_risk_domain"].map(spec_map)
    df_gain = classified[group == "gaining"]
    df_lose = classified[group == "losing"]
    print(f"=== {spec_name}: n gaining={len(df_gain)}, n losing={len(df_lose)} ===")
    h4_results_by_spec[spec_name] = h4_compare(df_gain, df_lose)
    print()

In [ ]:
# ── MODULE 5 : Cell 5.3 -- consolidated H4 summary across all three specs ──
h4_summary_rows = []
for spec_name, crit_results in h4_results_by_spec.items():
    for crit, res in crit_results.items():
        h4_summary_rows.append({
            "spec": spec_name, "criterion": crit,
            "gaining_value": res["gaining_value"], "losing_value": res["losing_value"],
            "test": res["test"], "statistic": res["statistic"],
            "p_value": res["p_value"], "p_display": res["p_display"],
            "ci_low": res["ci_low"], "ci_high": res["ci_high"],
            "adverse_direction": "MATCH" if res["adverse"] else "MISMATCH",
        })
h4_summary = pd.DataFrame(h4_summary_rows)
assert pd.api.types.is_float_dtype(h4_summary["p_value"])
print("H4 summary -- gaining- vs. losing-share groups, all specs:")
print(h4_summary.to_string(index=False))
print()

for spec_name, crit_results in h4_results_by_spec.items():
    n_adverse = sum(1 for res in crit_results.values() if res["adverse"])
    if n_adverse == 4:
        verdict = "SUPPORTED"
    elif n_adverse == 0:
        verdict = "NOT SUPPORTED"
    else:
        verdict = "PARTIALLY SUPPORTED"
    print(f"{spec_name}: H4 verdict = {verdict} ({n_adverse}/4 criteria adverse)")
print()
print("NOTE: B_share_change is the main specification because it matches H4's "
      "share-based wording ('risk categories gaining share'). A_count_trend and "
      "C_core_movers are robustness checks, not alternative main results.")

# Cross-check vs. the expected Spec B (main) values -- reported as computed,
# not adjusted to force a match.
EXPECTED_SPEC_B = {
    "H3a": (76.0, 18.7),
    "H3b": (56.2, 17.0),
    "H3c": (75.4, 33.1),
    "H3d": (25.36, 6.10),
}
print()
print("Cross-check vs. expected Spec B values (gaining, losing; tol=0.1):")
spec_b_results = h4_results_by_spec["B_share_change"]
for crit, (exp_g, exp_l) in EXPECTED_SPEC_B.items():
    actual = spec_b_results[crit]
    g_match = abs(actual["gaining_value"] - exp_g) < 0.1
    l_match = abs(actual["losing_value"] - exp_l) < 0.1
    verdict = "MATCH" if (g_match and l_match) else "MISMATCH"
    print(f"  {crit}: notebook gaining={actual['gaining_value']:.2f}, losing={actual['losing_value']:.2f} "
          f"vs. expected gaining={exp_g}, losing={exp_l} -> {verdict}")
    if verdict == "MISMATCH":
        print(f"    FLAGGED MISMATCH -- reported as-is, not adjusted to force agreement.")

# Cross-check vs. an independently-computed H3d bootstrap (seed 42, but a
# different RNG call order -- exact values are expected to differ slightly;
# flagged as-is if outside the expected envelope, not adjusted to force
# agreement).
EXPECTED_H3D_BOOT = {
    "B_share_change": {"diff": 19.3, "ci_low": 13.5, "ci_high": 25.4},
    "A_count_trend": {"diff": 17.4, "ci_low": 12.2, "ci_high": 23.3},
    "C_core_movers": {"diff": 20.0, "ci_low": 14.4, "ci_high": 26.4},
}
print()
print("Cross-check vs. expected H3d bootstrap ranges:")
for spec_name, expected in EXPECTED_H3D_BOOT.items():
    res_h3d = h4_results_by_spec[spec_name]["H3d"]
    diff_actual = res_h3d["statistic"]
    ci_lo_actual = res_h3d["ci_low"]
    ci_hi_actual = res_h3d["ci_high"]
    within_expected_ci = expected["ci_low"] <= diff_actual <= expected["ci_high"]
    excludes_zero = ci_lo_actual > 0
    verdict = "MATCH" if (within_expected_ci and excludes_zero) else "MISMATCH -- FLAGGED"
    print(f"  {spec_name}: notebook diff={diff_actual:.2f}, CI=[{ci_lo_actual:.2f}, {ci_hi_actual:.2f}] "
          f"vs. expected diff~{expected['diff']}, CI~[{expected['ci_low']}, {expected['ci_high']}] -> {verdict}")

h4_out_path = os.path.join(BATCH_DIR, "module5_h4_summary.csv")
h4_summary.to_csv(h4_out_path, index=False)
print(f"\nSaved {h4_out_path}")

## Module 4 robustness: entity classification (H3c, H3d, H4)

Re-runs H3c, H3d (deployer AND developer side), the new "unattributable
deployer" check, and H4 under three entity-definition specs, using the
HAND-REVIEWED entity codebook (`module4_entity_codebook_reviewed.csv`) --
not the unreviewed first-pass file from the cell above. Does not change
Modules 1-3, and does not modify the existing H3a-H3d/Module 5 cells --
those still report the original (Spec A, uncanonicalised) numbers exactly
as before; this section reruns the same underlying logic under the
alternative entity definitions, reusing `overdispersion_test()`,
`H4_SPECS`, `B_BOOT`/`rng`, and `h4_compare()`'s bootstrap approach from
Module 4/5 rather than duplicating them. (Not numbered as a top-level
module -- "Module 6" in the paper is the parked insurer-narrative
module; this is a robustness subsection of Module 4.)

Specs:
- **A_raw**: current behaviour exactly as in the H3c/H3d/H4 cells above --
  raw labels, no canonicalisation, no generic/unknown handling. Reported
  as a reproduction check against the existing figures.
- **A_canon**: same pooling as A_raw, but with `canonical_name` applied
  (e.g. Waymo -> Alphabet) -- isolates the effect of canonicalisation alone.
- **B_main**: generic/unknown labels replaced by a unique per-incident,
  per-role placeholder (so they never pool into a false "repeat entity");
  named entities canonicalised. This is the main spec.
- **C_named_only**: generic/unknown labels dropped entirely; named
  entities only, canonicalised.

`entity_class == "missing"` (a blank/unreviewed cell in the reviewed
codebook) is dropped in ALL FOUR variants -- so even A_raw/A_canon are not
expected to reproduce the original figures bit-for-bit if any such rows
exist; treat the reproduction check as approximate, not exact, and this
cell reports the gap explicitly rather than assuming it is zero.

In [ ]:
# ── Module 4 robustness (entity classification) : Cell 1 -- load the HAND-REVIEWED entity codebook ──
#
# This whole section reuses objects/functions built in Module 4 and
# Module 5 above (classified, domains, overdispersion_test, H4_SPECS,
# B_BOOT, rng, h4_compare) -- fail loudly and immediately if any are
# missing, rather than letting a later cell crash with a confusing
# NameError, which usually just means the notebook wasn't run top-to-bottom.
_REQUIRED_OBJECTS = {
    "classified": "Module 1", "domains": "Module 1",
    "overdispersion_test": "Module 4 (H3d)", "H4_SPECS": "Module 5 (Cell 5.1)",
    "B_BOOT": "Module 5 (Cell 5.2)", "rng": "Module 5 (Cell 5.2)",
    "h4_compare": "Module 5 (Cell 5.2)",
}
_missing_objects = [f"{name} (expected from {where})" for name, where in _REQUIRED_OBJECTS.items()
                     if name not in globals()]
if _missing_objects:
    raise NameError(
        "Required object(s) from earlier cells are missing: " + "; ".join(_missing_objects) + ". "
        "Run all from the top (Runtime -> Run all) before running this section -- "
        "it depends on Module 1, Module 4, and Module 5 having already executed."
    )

# Per instruction: if the reviewed file is not found, STOP and say so --
# do NOT fall back to the unreviewed module4_entity_codebook.csv.
REVIEWED_CODEBOOK_FILENAME = "module4_entity_codebook_reviewed.csv"
reviewed_codebook_path = os.path.join(BATCH_DIR, REVIEWED_CODEBOOK_FILENAME)

if not os.path.exists(reviewed_codebook_path):
    raise FileNotFoundError(
        f"Reviewed entity codebook not found at {reviewed_codebook_path}. "
        "This step requires the hand-reviewed "
        f"{REVIEWED_CODEBOOK_FILENAME} -- NOT falling back to the "
        "unreviewed module4_entity_codebook.csv. Upload the reviewed file "
        "to BATCH_DIR and re-run this cell."
    )

reviewed_codebook = pd.read_csv(reviewed_codebook_path)
print(f"Loaded {reviewed_codebook_path}: {len(reviewed_codebook)} rows")

REQUIRED_COLS = {"label", "entity_class", "canonical_name"}
missing_cols = REQUIRED_COLS - set(reviewed_codebook.columns)
if missing_cols:
    raise ValueError(f"Reviewed codebook is missing expected column(s): {missing_cols}")


def _normalize_entity_class(v):
    """entity_class is used exactly as reviewed -- this only normalizes
    how a blank/unreviewed cell is spelled (NaN, empty string, or the
    literal string "missing" all mean the same thing: drop this label)."""
    if pd.isna(v) or str(v).strip() == "":
        return "missing"
    v_norm = str(v).strip().lower()
    if v_norm not in {"named", "generic", "unknown", "missing"}:
        raise ValueError(f"Unexpected entity_class value in reviewed codebook: {v!r}")
    return v_norm


reviewed_codebook["entity_class_norm"] = reviewed_codebook["entity_class"].apply(_normalize_entity_class)
# canonical_name is used AS REVIEWED; only fall back to the raw label if a
# row genuinely has no canonical_name at all (shouldn't normally happen,
# since the Step-1 pre-fill always set one -- defensive, not a rewrite).
reviewed_codebook["canonical_name_norm"] = reviewed_codebook["canonical_name"].fillna(reviewed_codebook["label"])

print()
print("entity_class distribution (as reviewed):")
print(reviewed_codebook["entity_class_norm"].value_counts().to_string())

n_relabeled_missing = int((reviewed_codebook["entity_class_norm"] == "missing").sum())
print(f"\n{n_relabeled_missing} label(s) marked 'missing' -- dropped entirely in every spec below.")

label_info = {
    row["label"]: {"entity_class": row["entity_class_norm"], "canonical_name": row["canonical_name_norm"]}
    for _, row in reviewed_codebook.iterrows()
}

_labels_in_data = set(
    e for lst in classified["developer_entities"] for e in lst
) | set(
    e for lst in classified["deployer_entities"] for e in lst
)
_unreviewed_labels = _labels_in_data - set(label_info)
if _unreviewed_labels:
    print(f"\nWARNING: {len(_unreviewed_labels)} label(s) in developer_entities/deployer_entities "
          f"are NOT present in the reviewed codebook at all (treated as 'missing' -> dropped): "
          f"{sorted(_unreviewed_labels)[:20]}{'...' if len(_unreviewed_labels) > 20 else ''}")

In [ ]:
# ── Module 4 robustness (entity classification) : Cell 2 -- build the four entity-spec variants ──
ENTITY_SPECS = ["A_raw", "A_canon", "B_main", "C_named_only"]


def _label_lookup(label):
    info = label_info.get(label)
    if info is None:
        return "missing", label
    return info["entity_class"], info["canonical_name"]


def apply_entity_spec(entities, incident_id, role, spec):
    """Transforms one incident's raw entity list under one spec.

    'missing' labels are dropped in every spec. Named labels are pooled
    under their canonical_name in every spec except A_raw (raw label).
    Generic/unknown labels are handled per spec: pooled as-is (A_raw),
    pooled under canonical_name (A_canon -- same pooling behaviour as
    A_raw/current, just with canonicalisation applied), replaced by a
    unique per-incident-per-role placeholder so they can never look like
    a repeat entity (B_main), or dropped entirely (C_named_only).
    """
    out = []
    for e in entities:
        cls, canon = _label_lookup(e)
        if cls == "missing":
            continue
        if cls == "named":
            out.append(canon if spec != "A_raw" else e)
            continue
        # generic / unknown
        if spec == "A_raw":
            out.append(e)
        elif spec == "A_canon":
            out.append(canon)
        elif spec == "B_main":
            out.append(f"GENERIC_{incident_id}_{role}")
        elif spec == "C_named_only":
            continue
        else:
            raise ValueError(f"Unknown spec {spec!r}")
    return list(dict.fromkeys(out))


# classified_work holds one pair of columns per spec: deployer_entities__<spec>,
# developer_entities__<spec>. Built on a COPY so the original classified
# (and every cell above/H3a-H3d/Module 5 using it) is untouched.
classified_work = classified.copy()
for spec in ENTITY_SPECS:
    classified_work[f"developer_entities__{spec}"] = [
        apply_entity_spec(ents, iid, "developer", spec)
        for ents, iid in zip(classified_work["developer_entities"], classified_work["incident_id"])
    ]
    classified_work[f"deployer_entities__{spec}"] = [
        apply_entity_spec(ents, iid, "deployer", spec)
        for ents, iid in zip(classified_work["deployer_entities"], classified_work["incident_id"])
    ]

# Per-incident deployer-side entity_class list (SPEC-INDEPENDENT: classification
# is fixed by the reviewed codebook, not by how a spec represents the
# label downstream) -- used for the "unattributable deployer" check.
classified_work["deployer_classes"] = classified_work["deployer_entities"].apply(
    lambda ents: [c for c in (label_info.get(e, {"entity_class": "missing"})["entity_class"] for e in ents)
                  if c != "missing"]
)

print("Built entity-spec variants:", ENTITY_SPECS)
for spec in ENTITY_SPECS:
    n_dep = sum(len(x) > 0 for x in classified_work[f"deployer_entities__{spec}"])
    n_dev = sum(len(x) > 0 for x in classified_work[f"developer_entities__{spec}"])
    print(f"  {spec}: incidents with >=1 deployer entity = {n_dep}, "
          f">=1 developer entity = {n_dev}")

In [ ]:
# ── Module 4 robustness (entity classification) : Cell 3 -- metric helper functions (reused across specs/groups) ──
from scipy.stats import chi2_contingency


def pct_disjoint_metric(df, dev_col, dep_col):
    """H3c, single population: % disjoint among incidents with >=1 entity
    on BOTH sides under this spec's columns."""
    has_both = df.apply(lambda r: len(r[dev_col]) > 0 and len(r[dep_col]) > 0, axis=1)
    sub = df[has_both]
    n_both = len(sub)
    if n_both == 0:
        return {"n": 0, "pct_disjoint": float("nan")}
    disjoint = sub.apply(
        lambda r: {x.lower() for x in r[dev_col]}.isdisjoint({x.lower() for x in r[dep_col]}), axis=1
    )
    return {"n": n_both, "pct_disjoint": float(disjoint.mean() * 100)}


def unattributable_flags(deployer_classes_series):
    """True iff the incident has >=1 (non-missing) deployer label AND every
    one of them is generic/unknown. False (not True) if the deployer field
    is empty -- "no attribution info" is not the same claim as
    "unattributable because only generic labels were given"."""
    return deployer_classes_series.apply(
        lambda classes: len(classes) > 0 and all(c in ("generic", "unknown") for c in classes)
    )


def h3d_full_metrics(entities_series, n_total_incidents, label):
    """H3d, single population: dispersion ratio/Pearson/NB-alpha (via the
    Module 4 overdispersion_test), singleton share, and the incident-level
    (not mention-level) share involving an entity with >=5 incidents --
    same incident-level definition as the Module 4 H3d bug fix."""
    counts = Counter(e for lst in entities_series for e in lst)
    counts_arr = np.array(list(counts.values()), dtype=float)
    n_entities = len(counts_arr)
    if n_entities < 2:
        return {
            "n_entities": n_entities, "ratio": float("nan"), "p_value": float("nan"),
            "nb_alpha": float("nan"), "nb_alpha_ci_low": float("nan"), "nb_alpha_ci_high": float("nan"),
            "pct_singleton": float("nan"), "n_entities_ge5": 0, "pct_incidents_ge5": float("nan"),
        }
    result = overdispersion_test(counts_arr, label)
    n_singleton = int((counts_arr == 1).sum())
    pct_singleton = n_singleton / n_entities * 100
    heavy = {k for k, v in counts.items() if v >= 5}
    n_incidents_ge5 = sum(1 for lst in entities_series if any(e in heavy for e in lst))
    pct_incidents_ge5 = n_incidents_ge5 / n_total_incidents * 100 if n_total_incidents else float("nan")
    return {
        "n_entities": n_entities, "ratio": result["ratio"], "p_value": result["p_value"],
        "nb_alpha": result["nb_alpha"],
        "nb_alpha_ci_low": result["nb_alpha_ci_low"], "nb_alpha_ci_high": result["nb_alpha_ci_high"],
        "pct_singleton": pct_singleton, "n_entities_ge5": len(heavy),
        "pct_incidents_ge5": pct_incidents_ge5,
    }


def _entity_counts_arr(df, col):
    counts = Counter(e for entities in df[col] for e in entities)
    return np.array(list(counts.values()), dtype=float)


def h3c_between_groups(df_gain, df_lose, dev_col, dep_col):
    """H3c, gaining-vs-losing: mirrors Module 5's h4_compare H3c block,
    parameterized by which spec's entity columns to read."""
    def subset(df):
        has_both = df.apply(lambda r: len(r[dev_col]) > 0 and len(r[dep_col]) > 0, axis=1)
        sub = df[has_both].copy()
        sub["disjoint"] = sub.apply(
            lambda r: {x.lower() for x in r[dev_col]}.isdisjoint({x.lower() for x in r[dep_col]}), axis=1
        )
        return sub

    g, l = subset(df_gain), subset(df_lose)
    table = pd.DataFrame({
        "gaining": [int(g["disjoint"].sum()), int((~g["disjoint"]).sum())],
        "losing": [int(l["disjoint"].sum()), int((~l["disjoint"]).sum())],
    }, index=["disjoint", "overlap"])
    chi2_stat, p, _, _ = chi2_contingency(table.T)
    pct_g = table.loc["disjoint", "gaining"] / table["gaining"].sum() * 100 if table["gaining"].sum() else float("nan")
    pct_l = table.loc["disjoint", "losing"] / table["losing"].sum() * 100 if table["losing"].sum() else float("nan")
    return {
        "n_gain": len(g), "n_lose": len(l), "gaining_value": pct_g, "losing_value": pct_l,
        "statistic": chi2_stat, "p_value": p, "adverse": pct_g > pct_l,
    }


def unattributable_between_groups(df_gain, df_lose):
    fg = unattributable_flags(df_gain["deployer_classes"])
    fl = unattributable_flags(df_lose["deployer_classes"])
    table = pd.DataFrame({
        "gaining": [int(fg.sum()), int((~fg).sum())],
        "losing": [int(fl.sum()), int((~fl).sum())],
    }, index=["unattributable", "attributable"])
    chi2_stat, p, _, _ = chi2_contingency(table.T)
    return {
        "n_gain": len(fg), "n_lose": len(fl),
        "gaining_value": float(fg.mean() * 100), "losing_value": float(fl.mean() * 100),
        "statistic": chi2_stat, "p_value": p, "adverse": fg.mean() > fl.mean(),
    }


def h3d_bootstrap_between_groups(df_gain, df_lose, col, b=None):
    """H3d, gaining-vs-losing: same bootstrap-diff-in-dispersion-ratio
    approach as Module 5 Cell 5.2's H3d block (incident resampling, not
    entity resampling), parameterized by column so it works for either
    deployer_entities__<spec> or developer_entities__<spec>. Reuses the
    shared B_BOOT/rng defined in Module 5 Cell 5.2 for one continuous,
    reproducible draw stream across this whole module."""
    b = B_BOOT if b is None else b
    counts_g = _entity_counts_arr(df_gain, col)
    counts_l = _entity_counts_arr(df_lose, col)
    if len(counts_g) < 2 or len(counts_l) < 2:
        return {"gaining_value": float("nan"), "losing_value": float("nan"), "statistic": float("nan"),
                "p_value": float("nan"), "ci_low": float("nan"), "ci_high": float("nan"), "adverse": False}
    res_g = overdispersion_test(counts_g, f"{col} gaining group")
    res_l = overdispersion_test(counts_l, f"{col} losing group")
    observed_diff = res_g["ratio"] - res_l["ratio"]

    gain_idx = df_gain.index.to_numpy()
    lose_idx = df_lose.index.to_numpy()
    diffs = np.empty(b)
    for i in range(b):
        boot_gain = df_gain.loc[rng.choice(gain_idx, size=len(gain_idx), replace=True)]
        boot_lose = df_lose.loc[rng.choice(lose_idx, size=len(lose_idx), replace=True)]
        cg, cl = _entity_counts_arr(boot_gain, col), _entity_counts_arr(boot_lose, col)
        if len(cg) < 2 or len(cl) < 2:
            diffs[i] = np.nan
            continue
        diffs[i] = (cg.var(ddof=1) / cg.mean()) - (cl.var(ddof=1) / cl.mean())
    diffs = diffs[~np.isnan(diffs)]
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5]) if len(diffs) else (float("nan"), float("nan"))
    n_le_zero = int((diffs <= 0).sum())
    p_value = (n_le_zero / b) if n_le_zero > 0 else 1 / b
    return {
        "gaining_value": res_g["ratio"], "losing_value": res_l["ratio"],
        "statistic": observed_diff, "p_value": p_value,
        "ci_low": ci_low, "ci_high": ci_high, "adverse": ci_low > 0,
    }

In [ ]:
# ── Module 4 robustness (entity classification) : Cell 4 -- run all specs, both single-population and H4 gaining-vs-losing ──
entity_robustness_rows = []
n_total = len(classified_work)

# --- Single-population metrics per entity spec (no H4 grouping) ---
for spec in ENTITY_SPECS:
    dev_col, dep_col = f"developer_entities__{spec}", f"deployer_entities__{spec}"

    h3c = pct_disjoint_metric(classified_work, dev_col, dep_col)
    entity_robustness_rows.append({
        "metric": "H3c_pct_disjoint", "entity_spec": spec, "group": "overall",
        "n": h3c["n"], "statistic": h3c["pct_disjoint"], "p_value": None, "ci_low": None, "ci_high": None,
    })

    flags = unattributable_flags(classified_work["deployer_classes"])
    entity_robustness_rows.append({
        "metric": "unattributable_deployer_pct", "entity_spec": spec, "group": "overall",
        "n": n_total, "statistic": float(flags.mean() * 100), "p_value": None, "ci_low": None, "ci_high": None,
    })

    for side, col in [("deployer", dep_col), ("developer", dev_col)]:
        h3d = h3d_full_metrics(classified_work[col], n_total, f"{spec} {side}")
        entity_robustness_rows.append({
            "metric": f"H3d_{side}_dispersion_ratio", "entity_spec": spec, "group": "overall",
            "n": h3d["n_entities"], "statistic": h3d["ratio"], "p_value": h3d["p_value"],
            "ci_low": h3d["nb_alpha_ci_low"], "ci_high": h3d["nb_alpha_ci_high"],
        })
        entity_robustness_rows.append({
            "metric": f"H3d_{side}_pct_singleton", "entity_spec": spec, "group": "overall",
            "n": h3d["n_entities"], "statistic": h3d["pct_singleton"], "p_value": None, "ci_low": None, "ci_high": None,
        })
        entity_robustness_rows.append({
            "metric": f"H3d_{side}_pct_incidents_ge5", "entity_spec": spec, "group": "overall",
            "n": h3d["n_entities_ge5"], "statistic": h3d["pct_incidents_ge5"], "p_value": None,
            "ci_low": None, "ci_high": None,
        })

# --- Gaining-vs-losing (H4) metrics, per entity spec x per H4 grouping spec ---
# H4_SPECS (A_count_trend / B_share_change / C_core_movers) is reused as-is
# from Module 5 Cell 5.1 -- not rebuilt here.
for h4_spec_name, h4_spec_map in H4_SPECS.items():
    group = classified_work["mit_risk_domain"].map(h4_spec_map)
    df_gain = classified_work[group == "gaining"]
    df_lose = classified_work[group == "losing"]

    for spec in ENTITY_SPECS:
        dev_col, dep_col = f"developer_entities__{spec}", f"deployer_entities__{spec}"

        h3c_g = h3c_between_groups(df_gain, df_lose, dev_col, dep_col)
        entity_robustness_rows.append({
            "metric": "H4_H3c_pct_disjoint", "entity_spec": spec, "group": h4_spec_name,
            "n": f"{h3c_g['n_gain']}/{h3c_g['n_lose']}", "statistic": h3c_g["statistic"],
            "p_value": h3c_g["p_value"], "ci_low": None, "ci_high": None,
            "gaining_value": h3c_g["gaining_value"], "losing_value": h3c_g["losing_value"],
            "adverse_direction": "MATCH" if h3c_g["adverse"] else "MISMATCH",
        })

        unattr_g = unattributable_between_groups(df_gain, df_lose)
        entity_robustness_rows.append({
            "metric": "H4_unattributable_deployer_pct", "entity_spec": spec, "group": h4_spec_name,
            "n": f"{unattr_g['n_gain']}/{unattr_g['n_lose']}", "statistic": unattr_g["statistic"],
            "p_value": unattr_g["p_value"], "ci_low": None, "ci_high": None,
            "gaining_value": unattr_g["gaining_value"], "losing_value": unattr_g["losing_value"],
            "adverse_direction": "MATCH" if unattr_g["adverse"] else "MISMATCH",
        })

        for side, col in [("deployer", dep_col), ("developer", dev_col)]:
            h3d_g = h3d_bootstrap_between_groups(df_gain, df_lose, col)
            entity_robustness_rows.append({
                "metric": f"H4_H3d_{side}_dispersion_ratio_diff", "entity_spec": spec, "group": h4_spec_name,
                "n": f"{len(df_gain)}/{len(df_lose)}", "statistic": h3d_g["statistic"],
                "p_value": h3d_g["p_value"], "ci_low": h3d_g["ci_low"], "ci_high": h3d_g["ci_high"],
                "gaining_value": h3d_g["gaining_value"], "losing_value": h3d_g["losing_value"],
                "adverse_direction": "MATCH" if h3d_g["adverse"] else "MISMATCH",
            })

entity_robustness_results = pd.DataFrame(entity_robustness_rows)
print(f"Entity-robustness results: {len(entity_robustness_results)} rows "
      f"({len(ENTITY_SPECS)} entity specs x overall + x {len(H4_SPECS)} H4 groupings).")
print(entity_robustness_results.to_string(index=False))

In [ ]:
# ── Module 4 robustness (entity classification) : Cell 5 -- Spec A reproduction check, mismatch flags, save ──
print("Reproduction check -- Spec A_raw vs. the original H3c/H3d/H4 cells above:")
print("(approximate, not exact, if any label was marked entity_class='missing' "
      "in the reviewed codebook; the original cells never dropped anything)")
print()

EXPECTED_ORIGINAL = {
    ("H3c_pct_disjoint", "overall"): 58.0,
    ("H3d_deployer_dispersion_ratio", "overall"): 21.4,
}
for (metric, grp), expected in EXPECTED_ORIGINAL.items():
    row = entity_robustness_results[
        (entity_robustness_results["metric"] == metric) & (entity_robustness_results["entity_spec"] == "A_raw")
        & (entity_robustness_results["group"] == grp)
    ]
    if row.empty:
        print(f"  {metric} ({grp}): no matching row found")
        continue
    actual = row["statistic"].iloc[0]
    diff = abs(actual - expected)
    verdict = "MATCH" if diff < 0.5 else f"MISMATCH (diff={diff:.2f})"
    print(f"  {metric} ({grp}): A_raw={actual:.2f} vs. original={expected} -> {verdict}")

EXPECTED_H4_B_MAIN = {
    "H4_H3c_pct_disjoint": (75.4, 33.1),
    "H4_H3d_deployer_dispersion_ratio_diff": (25.4, 6.1),
}
print()
print("Reproduction check -- Spec A_raw, H4 B_share_change grouping, vs. the "
      "original Module 5 EXPECTED_SPEC_B values:")
for metric, (exp_g, exp_l) in EXPECTED_H4_B_MAIN.items():
    row = entity_robustness_results[
        (entity_robustness_results["metric"] == metric) & (entity_robustness_results["entity_spec"] == "A_raw")
        & (entity_robustness_results["group"] == "B_share_change")
    ]
    if row.empty:
        print(f"  {metric}: no matching row found")
        continue
    g_actual, l_actual = row["gaining_value"].iloc[0], row["losing_value"].iloc[0]
    verdict = "MATCH" if abs(g_actual - exp_g) < 0.5 and abs(l_actual - exp_l) < 0.5 else "MISMATCH"
    print(f"  {metric}: A_raw gaining={g_actual:.2f}/losing={l_actual:.2f} "
          f"vs. original gaining={exp_g}/losing={exp_l} -> {verdict}")

# --- Flag every H4 comparison whose direction differs from Spec A_raw ---
print()
print("H4 comparisons whose adverse direction differs from Spec A_raw (same metric x group):")
h4_rows = entity_robustness_results[entity_robustness_results["metric"].str.startswith("H4_")]
flagged = []
for (metric, grp), sub in h4_rows.groupby(["metric", "group"]):
    a_row = sub[sub["entity_spec"] == "A_raw"]
    if a_row.empty or pd.isna(a_row["adverse_direction"].iloc[0]):
        continue
    a_direction = a_row["adverse_direction"].iloc[0]
    for _, r in sub[sub["entity_spec"] != "A_raw"].iterrows():
        if pd.notna(r["adverse_direction"]) and r["adverse_direction"] != a_direction:
            flagged.append((metric, grp, r["entity_spec"], a_direction, r["adverse_direction"]))
if flagged:
    for metric, grp, spec, a_dir, spec_dir in flagged:
        print(f"  {metric} / {grp}: A_raw={a_dir}, {spec}={spec_dir}  <-- DIRECTION DIFFERS")
else:
    print("  (none -- every spec agrees with A_raw on direction for every H4 comparison)")

entity_robustness_out_path = os.path.join(BATCH_DIR, "module4_entity_robustness_results.csv")
entity_robustness_results.to_csv(entity_robustness_out_path, index=False)
print(f"\nSaved {entity_robustness_out_path}")